# AI Technologies (Basic Course) · Practical notebook
## Computer vision for UAV and UGV tasks in Google Colab: from a video frame to detection and tracking

**Kruty Heroes Military Institute of Telecommunications and Information Technology · Department of Special Information Systems and Robotic Complexes**

The notebook accompanies the five topics of the 40-hour programme. Step by step you take a **real** video recorded from a UAV, decompose it into frames, study the image as a data array, degrade and restore it, extract classical features, train a classifier of thermal objects, detect warm objects and track a moving vehicle with a Kalman filter.

**Result of the work:** a reproducible project with code, figures, tables of metrics and a short final report.

### How to work with the notebook
1. Open the file in [Google Colab](https://colab.research.google.com/) (*File → Upload notebook*) and save your own copy (*File → Save a copy in Drive*).
2. The base route runs on a CPU. A GPU (*Runtime → Change runtime type → T4 GPU*) speeds up only the optional blocks MobileNet, YOLO and ByteTrack.
3. Run the cells from top to bottom. Every block starts with a text cell: **Purpose**, **Parameters**, **What to look at**. The parameters are set in the forms on the right side of the code cells; after a change, run the cell again.
4. After a restart of the runtime run the cells again from cell 0.1.
5. Every figure and table is also saved in the folder `outputs` of the project.

### Data
All demonstrations use real open data sets with manual annotation (sources and licences are at the end). Only the files that a block needs are downloaded, the cache is kept between the blocks.

| Data set | What it contains | Where it is used |
|---|---|---|
| **M3OT** | visible and thermal image sequences of a DJI Mavic 3T UAV, 100–120 m, 10 frames/s, annotated vehicle tracks | Topic 1 (video), 3.3 (camera motion), 5.2 and 5.4 (tracking) |
| **VisDrone** | visible images of camera drones over cities with boxes of pedestrians and vehicles | Topics 2 and 3, 5.3 (YOLO) |
| **HIT-UAV** | thermal (LWIR) images of a DJI Matrice M210 V2, 60–130 m, day and night, boxes of persons, cars and bicycles | Topic 4 (classification), 5.1 (warm objects) |

### Map of the course

| Topic | Practical work in this notebook | Control result |
|---|---|---|
| **1. Google Colab and video data** | project folders, a real UAV video fragment, decomposition into frames with a given sampling rate | video file, table of time stamps, motion-activity plot |
| **2. Digital image** | colour channels, histograms, degradations of the video link, noise filters, CLAHE, letterbox | PSNR/SSIM tables and curves, object size after scaling |
| **3. Classical features** | Sobel, Canny, contours, ORB, homography, compensation of camera motion | edge-density curves, robustness of ORB to rotation and scale |
| **4. Machine learning** | data set of thermal patches, HOG + SVM, hyperparameter map, learning curve, MobileNet (optional) | verified classifier, confusion matrix, comparison of features |
| **5. Detection and tracking** | mask and boxes of warm objects, precision–recall curves day/night, Kalman filter, YOLO11 and ByteTrack (optional) | detection metrics, tracking error and tuning curves |

Schedule of the programme: 10 hours of lectures, 8 hours of group classes and 22 hours of practical classes (training plan AIT-BC, 2026).

---
## 0. Preparation of the runtime

### 0.1. Libraries, random seed and project folders

**Purpose.** Check the runtime, fix the random seed and create the project structure:

| Folder | Content |
|---|---|
| `data` | video files and other source data of the project |
| `frames` | frames extracted from the video |
| `models` | trained models |
| `outputs` | figures, tables (CSV) and the final report (JSON) |

**Parameters.**
- `SEED` — seed of the random number generator. The same seed gives the same noise, the same data split and the same results after a restart.
- `USE_GOOGLE_DRIVE` — keep the project on Google Drive (it survives the end of the Colab session) instead of the temporary disk `/content`.

**What to look at.** The table of versions: it is saved in `outputs/environment.json`, because results of computer vision depend on library versions.

In [ ]:
#@title 0.1 Runtime, random seed and project folders { display-mode: "form" }
SEED = 42 #@param {type:"integer"}
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}

import sys, os, time, json, math, platform, shutil, subprocess, importlib, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

for module, package in {"sklearn": "scikit-learn", "skimage": "scikit-image"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
import sklearn, skimage

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

random_state = np.random.default_rng(SEED)
np.random.seed(SEED)

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/AITBC/practical_course"
else:
    PROJECT = os.path.join("/content" if IN_COLAB else os.getcwd(), "aitbc_practical_course")
DIRS = {name: os.path.join(PROJECT, name) for name in ["data", "frames", "models", "outputs"]}
for path in DIRS.values():
    os.makedirs(path, exist_ok=True)
OUT = DIRS["outputs"]

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "font.size": 10, "axes.titlesize": 11,
                     "axes.labelsize": 10, "legend.fontsize": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b", "#17becf"]


def save_figure(fig, name):
    # save a figure in outputs and show it
    fig.savefig(os.path.join(OUT, name), bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


def show_images(images, titles, name, cols=None, cmap="gray", width=4.2):
    # grid of images with titles
    cols = cols or len(images)
    rows = (len(images) + cols - 1) // cols
    h, w = images[0].shape[:2]
    fig, axes = plt.subplots(rows, cols, figsize=(width * cols, width * rows * h / w + 0.4 * rows), squeeze=False)
    for ax, im, title in zip(axes.flat, images, titles):
        ax.imshow(im, cmap=cmap if im.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    for ax in list(axes.flat)[len(images):]:
        ax.axis("off")
    fig.tight_layout()
    save_figure(fig, name)


ENVIRONMENT = {"Python": platform.python_version(), "NumPy": np.__version__, "OpenCV": cv2.__version__,
               "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "scikit-image": skimage.__version__,
               "Matplotlib": matplotlib.__version__, "Google Colab": "yes" if IN_COLAB else "no"}
with open(os.path.join(OUT, "environment.json"), "w") as fh:
    json.dump(ENVIRONMENT, fh, indent=2)
display(pd.DataFrame(ENVIRONMENT.items(), columns=["Component", "Version"]).set_index("Component"))
print("Project folder:", PROJECT)

### 0.2. Course library and real data sets

**Purpose.** Write the course library `aitbc` (version 2.1.0, its source is embedded in the cell) and connect the data sets HIT-UAV, VisDrone and M3OT. At this step only the annotation files and the directories of the archives are read (about 10 MB, 15–40 s). Images are downloaded later, when a block needs them.

**Parameters.** `CACHE_ON_DRIVE` — keep the download cache on Google Drive (`MyDrive/AITBC/data_cache`), so that a new session does not download the files again. The same cache is used by the other notebooks of the course.

In [ ]:
#@title 0.2 Course library aitbc and data sets { display-mode: "form" }
CACHE_ON_DRIVE = False #@param {type:"boolean"}

LIB_SOURCE = r'''"""
aitbc - course library of "AI Technologies (Basic Course)" (AIT-BC).

Kruty Heroes Military Institute of Telecommunications and Information Technology,
Kyiv, 2026.

All demonstrations of the course use REAL open data sets recorded by UAVs and
ground vehicles. Only the files that a block needs are downloaded: large archives
are read selectively with HTTP range requests, and every file is cached in
DATA_ROOT (a Google Drive folder keeps the cache between sessions).

Data sets
---------
HIT-UAV   2898 thermal (LWIR) images 640x512 and 5 thermal videos. DJI Matrice
          M210 V2 with a Zenmuse XT2 camera (25 mm lens, field of view 25 x 20 deg),
          flight altitude 60-130 m, camera angle 30-90 deg, day and night. Boxes of
          Person, Car, Bicycle, OtherVehicle, DontCare.
          Suo J. et al. Scientific Data 10, 227 (2023). Licence CC BY 4.0.
Kust4K    4024 pixel-aligned visible (RGB) and thermal (TIR) image pairs 640x512
          from a hexacopter over urban roads, day and night; pixel labels of 8 classes.
          Scientific Data 12, 1701 (2025). Licence CC BY 4.0.
VisDrone  RGB images of drones over cities (VisDrone2019-DET validation part,
          548 images) with boxes of pedestrians, vehicles and other objects.
          Zhu P. et al. IEEE TPAMI 44(11), 7380-7399 (2022).
GOOSE     camera images of a ground vehicle on roads, field tracks, in forests and on
          a training area, with pixel labels of 64 classes (validation part, 962 images).
          IEEE ICRA 2024. Licence CC BY-SA 4.0.
M3OT      synchronised visible and thermal image sequences of two DJI Mavic 3T drones
          (100-120 m, 10 fps, 640x512, day, dusk, night) with vehicle tracks.
          Nie Z. et al. Scientific Data 12, 1927 (2025). Licence CC BY 4.0.

Units: metres, seconds, degrees. Images are NumPy arrays H x W (grey) or H x W x 3 (RGB).
"""

__version__ = "2.1.0"

import csv
import io
import json
import math
import os
import struct
import time
import urllib.error
import urllib.request
import zlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import cv2

DATA_ROOT = os.environ.get("AITBC_DATA") or (
    "/content/aitbc_data" if os.path.isdir("/content") else os.path.abspath("aitbc_data"))


def set_data_root(path):
    """Set the folder of the download cache (for example a folder on Google Drive)."""
    global DATA_ROOT
    DATA_ROOT = os.path.abspath(path)
    os.makedirs(DATA_ROOT, exist_ok=True)
    return DATA_ROOT


def _cache(*parts):
    p = os.path.join(DATA_ROOT, *parts)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


def cache_size():
    """Number of files and size of the download cache, MB."""
    n, s = 0, 0
    for dp, _, fn in os.walk(DATA_ROOT):
        for f in fn:
            n += 1
            s += os.path.getsize(os.path.join(dp, f))
    return n, s / 1e6


# =============================================================================
# 1. Downloading
# =============================================================================
class _HTTPSRedirect(urllib.request.HTTPRedirectHandler):
    """Follow redirects, keep the Range header and upgrade http:// targets to https://."""

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if newurl.startswith("http://"):
            newurl = "https://" + newurl[len("http://"):]
        new = super().redirect_request(req, fp, code, msg, headers, newurl)
        if new is not None and req.has_header("Range"):
            new.add_header("Range", req.get_header("Range"))
        return new


_OPENER = urllib.request.build_opener(_HTTPSRedirect)
_OPENER.addheaders = [("User-Agent", "aitbc-course/2.0")]


def _urlopen(url, headers=None, timeout=60, retries=4):
    """Open url with retries; a new Request object is built for every attempt."""
    err = None
    for k in range(retries):
        try:
            return _OPENER.open(urllib.request.Request(url, headers=dict(headers or {})), timeout=timeout)
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404, 416):
                raise
            err = e
        except Exception as e:                      # network error: retry after a pause
            err = e
        time.sleep(1.5 * (k + 1))
    raise err


def fetch(url, path, timeout=120):
    """Download url to path; nothing is done when the file is already in the cache."""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".part"
    with _urlopen(url, timeout=timeout) as r, open(tmp, "wb") as fh:
        while True:
            b = r.read(1 << 20)
            if not b:
                break
            fh.write(b)
    os.replace(tmp, path)
    return path


def fetch_many(jobs, threads=8):
    """jobs: list of (url, path); parallel download, returns the list of paths."""
    with ThreadPoolExecutor(threads) as ex:
        return list(ex.map(lambda j: fetch(*j), jobs))


class _RangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests (reads the directory of a remote zip)."""

    def __init__(self, url, chunk=1 << 17, timeout=60):
        self.url, self.chunk, self.timeout = url, chunk, timeout
        r = _urlopen(url, {"Range": "bytes=0-0"}, timeout)
        cr = r.headers.get("Content-Range")
        r.read()
        if r.status != 206 or not cr:
            raise IOError(f"the server of {url} does not support range requests")
        self.size = int(cr.split("/")[-1])
        self.pos, self.buf_start, self.buf = 0, 0, b""

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        if not (self.buf_start <= self.pos and self.pos + n <= self.buf_start + len(self.buf)):
            start = self.pos
            end = min(self.size, start + max(n, self.chunk)) - 1
            self.buf = _urlopen(self.url, {"Range": f"bytes={start}-{end}"}, self.timeout).read()
            self.buf_start = start
        off = self.pos - self.buf_start
        out = self.buf[off:off + n]
        self.pos += len(out)
        return out

    def readinto(self, b):
        d = self.read(len(b))
        b[:len(d)] = d
        return len(d)


class RemoteZip:
    """Selective reading of members of a large remote zip archive: only the central
    directory and the requested members are transferred; members are cached on disk."""

    def __init__(self, url, cache_name):
        import zipfile
        self.url, self.cache_name = url, cache_name
        idx_path = _cache("_index", cache_name + ".json")
        if os.path.exists(idx_path):
            with open(idx_path) as fh:
                self.index = json.load(fh)
        else:
            zf = zipfile.ZipFile(_RangeFile(url))
            self.index = {i.filename: (i.header_offset, i.compress_size, i.file_size, i.compress_type)
                          for i in zf.infolist() if not i.filename.endswith("/")}
            with open(idx_path, "w") as fh:
                json.dump(self.index, fh)

    def namelist(self):
        return list(self.index)

    def _fetch_member(self, name):
        path = _cache(self.cache_name, name)
        if os.path.exists(path):
            return path
        off, csize, usize, ctype = self.index[name]
        raw = _urlopen(self.url, {"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}).read()
        if raw[:4] != b"PK\x03\x04":
            raise IOError(f"bad local header of {name}")
        n, m = struct.unpack("<HH", raw[26:30])
        data = raw[30 + n + m: 30 + n + m + csize]
        if ctype == 8:
            data = zlib.decompressobj(-15).decompress(data)
        elif ctype != 0:
            raise IOError(f"unsupported compression {ctype} in {name}")
        if len(data) != usize:
            raise IOError(f"incomplete member {name}")
        with open(path + ".part", "wb") as fh:
            fh.write(data)
        os.replace(path + ".part", path)
        return path

    def get(self, names, threads=8):
        """Download members (a name or a list of names); returns local path(s)."""
        single = isinstance(names, str)
        names = [names] if single else list(names)
        with ThreadPoolExecutor(threads) as ex:
            paths = list(ex.map(self._fetch_member, names))
        return paths[0] if single else paths


def _rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# =============================================================================
# 2. Data sets
# =============================================================================
class HITUAV:
    """HIT-UAV: thermal images and videos of a DJI Matrice M210 V2 with a Zenmuse XT2.

    File name T_HHH_AA_W_NNNNN: T = 0 day / 1 night, HHH = flight altitude, m,
    AA = camera angle, deg (90 = vertical view), W = 0 sunny / 1 rainy.
    Images are 8-bit white-hot (warm = bright) after the automatic gain control
    of the camera: brightness is a relative, not a calibrated, temperature."""

    PALETTE = "white-hot"
    BASE = "https://raw.githubusercontent.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/main/"
    CLASSES = ("Person", "Car", "Bicycle", "OtherVehicle", "DontCare")
    COLORS = {"Person": (0, 229, 255), "Car": (255, 235, 59), "Bicycle": (118, 255, 3),
              "OtherVehicle": (255, 152, 0), "DontCare": (190, 190, 190)}
    CAMERA = dict(model="DJI Zenmuse XT2, uncooled VOx microbolometer", width=640, height=512,
                  focal_mm=25.0, pixel_pitch_um=17.0, hfov_deg=25.0, vfov_deg=20.0, band_um="7.5-13.5")
    EXAMPLES = ('1_60_30_0_00047.jpg', '0_60_60_0_01734.jpg', '1_70_90_0_07776.jpg', '0_100_50_0_08162.jpg', '1_120_90_0_06161.jpg', '0_130_40_0_05603.jpg', '1_90_70_0_04990.jpg', '0_110_80_0_08518.jpg')
    VIDEOS = {"60m-30_1.mov": "60 m, camera angle 30 deg: people on a sports ground",
              "60m-40_3.mov": "60 m, 40 deg: cars on a parking lot",
              "70m-90_1.mov": "70 m, 90 deg (vertical view): people on a square",
              "120m-30_3.mov": "120 m, 30 deg: road along buildings",
              "130m-60_3.mov": "130 m, 60 deg: parking lot and roads"}

    def __init__(self, splits=("train", "val", "test")):
        import pandas as pd
        imgs, objs = [], []
        jobs = [(self.BASE + f"normal_json/annotations/{sp}.json", _cache("hituav", "annotations", sp + ".json"))
                for sp in splits]
        for sp, path in zip(splits, fetch_many(jobs, 3)):
            with open(path) as fh:
                d = json.load(fh)
            cat = {c["id"]: c["name"] for c in d["categories"]}
            id2f = {}
            for im in d["images"]:
                fn = im.get("filename") or im.get("file_name")
                id2f[im["id"]] = fn
                t, h, a, w = fn.split("_")[:4]
                imgs.append(dict(file=fn, split=sp, night=t == "1", altitude=int(h), angle=int(a), rainy=w == "1"))
            for an in d.get("annotation", d.get("annotations", [])):
                x, y, w, h = an["bbox"]
                objs.append(dict(ann_id=an["id"], file=id2f[an["image_id"]], cls=cat[an["category_id"]],
                                 x1=float(x), y1=float(y), x2=float(x + w), y2=float(y + h)))
        # oriented boxes [xc, yc, w, h, angle]: true width and length of rotated objects
        rjobs = [(self.BASE + f"rotate_json/annotations/{sp}.json", _cache("hituav", "annotations", "rot_" + sp + ".json"))
                 for sp in splits]
        rot = {}
        for path in fetch_many(rjobs, 3):
            with open(path) as fh:
                d = json.load(fh)
            for an in d.get("annotations", d.get("annotation", [])):
                b = an["bbox"]
                rot[an["id"]] = (min(b[2], b[3]), max(b[2], b[3]), b[4] if len(b) > 4 else 0.0)
        self.images = pd.DataFrame(imgs).sort_values("file").reset_index(drop=True)
        o = pd.DataFrame(objs)
        o["w"], o["h"] = o.x2 - o.x1, o.y2 - o.y1
        o["width_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[0])     # oriented box: short side
        o["length_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[1])    # oriented box: long side
        self.objects = o.merge(self.images[["file", "night", "altitude", "angle"]], on="file", how="left")
        self._split = dict(zip(self.images.file, self.images.split))
        self._groups = {f: g for f, g in self.objects.groupby("file")}
        counts = self.objects.pivot_table(index="file", columns="cls", values="x1", aggfunc="count", fill_value=0)
        self.images = self.images.merge(counts, left_on="file", right_index=True, how="left").fillna(0)

    def _url(self, f):
        return self.BASE + f"normal_json/{self._split[f]}/{f}"

    def path(self, file):
        return fetch(self._url(file), _cache("hituav", self._split[file], file))

    def prefetch(self, files, threads=8):
        return fetch_many([(self._url(f), _cache("hituav", self._split[f], f)) for f in files], threads)

    def load(self, file):
        """Thermal image (H x W uint8) and the list of annotated objects."""
        img = cv2.imread(self.path(file), cv2.IMREAD_GRAYSCALE)
        g = self._groups.get(file)
        objs = [] if g is None else [dict(cls=r.cls, box=[r.x1, r.y1, r.x2, r.y2]) for r in g.itertuples()]
        return img, objs

    def video(self, name):
        return fetch(self.BASE + "video_sample/" + name, _cache("hituav", "video", name))

    @staticmethod
    def video_geometry(name):
        """Flight altitude, m, and camera angle, deg, from a video name such as '60m-30_1.mov'."""
        alt, rest = name.split("m-")
        return int(alt), int(rest.split("_")[0])

    def gsd(self, altitude, angle=90):
        """Ground sampling distance at the image centre, m/px (formula of Lesson 1.1)."""
        rng = altitude / max(math.sin(math.radians(angle)), 1e-3)
        return 2 * rng * math.tan(math.radians(self.CAMERA["hfov_deg"]) / 2) / self.CAMERA["width"]


class Kust4K:
    """Kust4K: pixel-aligned RGB and thermal (TIR) pairs 640x512 of a hexacopter
    HET-3021C over urban roads, day (id ...D) and night (id ...N), with pixel labels.

    The thermal images are pseudo-colour (palette from purple = cold to yellow = warm);
    tir_gray is their luminance, a relative thermal brightness. The data set contains
    frames with SIMULATED failures (visible channel set to zero, thermal channel
    blurred) made by its authors; they are excluded from the curated lists below."""

    PALETTE = "pseudo-colour (purple = cold, yellow = warm)"
    URL = "https://ndownloader.figshare.com/files/{}"
    FILES = {"rgb": 55979147, "tir": 55979165, "seg": 55979159}
    CLASSES = ("unlabelled", "road", "building", "motorcycle", "car", "truck", "tree", "person", "traffic facilities")
    COLORS = np.array([(0, 0, 0), (150, 150, 150), (229, 57, 53), (255, 152, 0), (255, 235, 59),
                       (236, 64, 122), (46, 160, 67), (0, 229, 255), (30, 136, 229)], np.uint8)
    OBJECTS = (3, 4, 5, 7)
    CAMERA = dict(platform="hexacopter HET-3021C", tir_focal_mm=9.1, tir_pixel_um=12.0,
                  rgb="1920x1080, f = 8 mm, registered to the thermal image and resized to 640x512")
    DAY = ("00720D", "00719D", "00031D", "01944D", "00851D", "00580D", "00990D", "00355D", "02094D", "02293D")
    NIGHT = ("02566N", "02807N", "02921N", "03135N", "03685N", "03895N", "03780N", "04004N", "02867N", "02599N")

    def __init__(self):
        self._z = {}

    def _zip(self, key):
        if key not in self._z:
            self._z[key] = RemoteZip(self.URL.format(self.FILES[key]), "kust4k_" + key)
        return self._z[key]

    def prefetch(self, ids, threads=6):
        names = [i + ".png" for i in ids]
        with ThreadPoolExecutor(3) as ex:
            list(ex.map(lambda k: self._zip(k).get(names, threads), ("rgb", "tir", "seg")))

    def pair(self, fid):
        """dict(id, night, rgb, tir, tir_gray, labels) for a frame id such as '00720D'."""
        self.prefetch([fid])
        name = fid + ".png"
        rgb = _rgb(self._zip("rgb").get(name))
        tir = _rgb(self._zip("tir").get(name))
        lab = cv2.imread(self._zip("seg").get(name), cv2.IMREAD_UNCHANGED)
        return dict(id=fid, night=fid.endswith("N"), rgb=rgb, tir=tir,
                    tir_gray=cv2.cvtColor(tir, cv2.COLOR_RGB2GRAY), labels=lab)


class VisDrone:
    """VisDrone2019-DET validation part: RGB images of drones with object boxes.
    Annotation line: x, y, w, h, score, category, truncation, occlusion."""

    URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
    CLASSES = ("ignored region", "pedestrian", "people", "bicycle", "car", "van", "truck", "tricycle",
               "awning-tricycle", "bus", "motor", "others")
    COLORS = {"pedestrian": (0, 229, 255), "people": (0, 176, 255), "bicycle": (118, 255, 3), "car": (255, 235, 59),
              "van": (255, 193, 7), "truck": (255, 152, 0), "tricycle": (186, 104, 200),
              "awning-tricycle": (156, 39, 176), "bus": (244, 67, 54), "motor": (236, 64, 122)}

    CURATED = ('0000213_03920_d_0000243', '0000291_03801_d_0000887', '0000117_00112_d_0000087', '0000001_08414_d_0000013', '0000155_00401_d_0000001', '0000193_01705_d_0000112', '0000295_02400_d_0000033', '0000289_04601_d_0000834', '0000215_00000_d_0000256', '0000327_04001_d_0000731')

    def __init__(self):
        self.z = RemoteZip(self.URL, "visdrone_val")

    def names(self):
        return sorted(n.split("/")[-1][:-4] for n in self.z.namelist() if n.endswith(".jpg"))

    def _members(self, name):
        return [f"VisDrone2019-DET-val/images/{name}.jpg", f"VisDrone2019-DET-val/annotations/{name}.txt"]

    def prefetch(self, names, threads=8):
        self.z.get([m for n in names for m in self._members(n)], threads)

    def load(self, name):
        """RGB image and the list of objects (ignored regions and 'others' are skipped)."""
        pi, pa = self.z.get(self._members(name))
        objs = []
        with open(pa) as fh:
            for line in fh:
                v = line.strip().strip(",").split(",")
                if len(v) < 8:
                    continue
                x, y, w, h, _, c, trunc, occ = (int(float(s)) for s in v[:8])
                if c in (0, 11):
                    continue
                objs.append(dict(cls=self.CLASSES[c], box=[x, y, x + w, y + h], truncation=trunc, occlusion=occ))
        return _rgb(pi), objs


class M3OT:
    """M3OT: synchronised visible (RGB) and thermal (IR) image sequences of two DJI Mavic 3T
    drones flying at 100-120 m (10 frames per second, 640 x 512 px), day, dusk and night, with
    multi-object tracking annotation of vehicles (MOT format: frame, id, x, y, w, h, flag, class,
    confidence). The track identifiers are the same in the RGB and IR sequences of one drone.
    The two channels were aligned by the authors with an affine transform; a residual spatial
    offset of several pixels and a time offset of up to several frames remain (Lesson 1.4).

    The IR frames are 8-bit grey images in the BLACK-HOT palette (warm = dark). The data
    descriptor does not state the palette; it is established from the scene: in all nine
    sequences of drone 1 the road surface is darker than vegetation by day, at dusk and at night,
    while asphalt is, as a rule, the warmer surface (Lesson 1.3, task 1.3.4). Vehicles are
    bright, i.e. cooler than the road. to_white_hot() gives the white-hot convention of HIT-UAV."""

    PALETTE = "black-hot"
    URL = "https://ndownloader.figshare.com/files/52023875"
    CAMERA = dict(platform="DJI Mavic 3T", altitude_m="100-120", fps=10, width=640, height=512,
                  rgb="telephoto camera, f = 29.85 mm (35 mm equivalent 162 mm), frames registered to IR",
                  ir="uncooled thermal camera 640 x 512, f = 9.1 mm (35 mm equivalent 40 mm)")
    SEQUENCES = {"1-01": "day, suburban road", "1-02": "night, city crossroads", "1-03": "day, city road",
                 "1-04": "day, suburban highway", "1-05": "day, crossroads", "1-06": "dusk, highway",
                 "1-07": "day, crossroads", "1-08": "dusk, crossroads", "1-09": "day, city street with parked cars"}

    def __init__(self):
        self.z = RemoteZip(self.URL, "m3ot")
        self._seq = {}
        for n in self.z.namelist():
            parts = n.split("/")        # M3OT/<drone>/<rgb|ir>/<split>/<sequence>/img1/<frame>.PNG
            if len(parts) == 7 and parts[5] == "img1":
                self._seq.setdefault((parts[4].rstrip("T"), parts[2]), []).append(n)
        for k in self._seq:
            self._seq[k].sort()

    def sequences(self):
        return sorted({k[0] for k in self._seq})

    def n_frames(self, seq, channel="rgb"):
        return len(self._seq[(seq, channel)])

    def frame_names(self, seq, channel="rgb", start=0, count=None, step=1):
        names = self._seq[(seq, channel)]
        end = len(names) if count is None else min(len(names), start + count * step)
        return names[start:end:step]

    def prefetch(self, seq, channel="rgb", start=0, count=None, step=1, threads=8):
        return self.z.get(self.frame_names(seq, channel, start, count, step), threads)

    def frames(self, seq, channel="rgb", start=0, count=None, step=1):
        """List of RGB images (H x W x 3) of a sequence; IR frames are grey images in 3 equal channels."""
        return [_rgb(p) for p in self.prefetch(seq, channel, start, count, step)]

    def tracks(self, seq, channel="rgb"):
        """Annotation table: frame (from 1), id, x, y, w, h, cx, cy."""
        import pandas as pd
        names = self._seq[(seq, channel)]
        gt = names[0].rsplit("/img1/", 1)[0] + "/gt/gt.txt"
        g = pd.read_csv(self.z.get(gt), header=None, names=["frame", "id", "x", "y", "w", "h", "flag", "cls", "conf"])
        g["cx"], g["cy"] = g.x + g.w / 2, g.y + g.h / 2
        return g[["frame", "id", "x", "y", "w", "h", "cx", "cy"]]

    def objects(self, table, frame):
        """Objects of one frame in the format of draw_boxes."""
        t = table[table.frame == frame]
        return [dict(cls="vehicle", id=int(r.id), box=[r.x, r.y, r.x + r.w, r.y + r.h]) for r in t.itertuples()]


class GOOSE:
    """GOOSE: front camera of a ground vehicle (2048x1000) with pixel labels of 64 classes."""

    URL = "https://goose-dataset.de/storage/goose_2d_val.zip"
    TRAVERSABLE = ("asphalt", "cobble", "gravel", "soil", "low_grass", "moss", "leaves", "snow",
                   "road_marking", "pedestrian_crossing", "sidewalk", "bikeway")
    CONDITIONAL = ("high_grass", "crops", "debris", "tree_root")
    IGNORE = ("undefined", "sky", "ego_vehicle", "outlier")
    CURATED = (
        'labels/val/2022-07-22_flight/2022-07-22_flight__0078_1658494246011438339_labelids.png',
        'labels/val/2022-07-22_flight/2022-07-22_flight__0158_1658494465090456964_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0528_1661860582736903436_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0551_1661860697315889089_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0002_1663755194931038715_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0038_1663755353192423465_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0114_1677850404544550420_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0156_1677850669766430912_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0007_1670420722571435228_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0057_1670421449778652057_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0037_1670421230548919820_labelids.png',
        'labels/val/2023-05-15_neubiberg_rain/2023-05-15_neubiberg_rain__0650_1684157928693659670_labelids.png',
    )

    def __init__(self):
        self.z = RemoteZip(self.URL, "goose_val")
        self.names, self.colors = {}, {}
        with open(self.z.get("goose_label_mapping.csv"), newline="") as fh:
            for row in csv.DictReader(fh):
                k = int(row["label_key"])
                h = row["hex"].lstrip("#")
                self.names[k] = row["class_name"]
                self.colors[k] = tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))
        self.lut = np.zeros((256, 3), np.uint8)
        for k, c in self.colors.items():
            self.lut[k] = c
        self.id_of = {v: k for k, v in self.names.items()}

    def frames(self):
        return sorted(n for n in self.z.namelist() if n.endswith("_labelids.png"))

    @staticmethod
    def _image_member(label_member):
        return label_member.replace("labels/", "images/").replace("_labelids.png", "_windshield_vis.png")

    def prefetch(self, members, threads=6):
        self.z.get([m for lm in members for m in (self._image_member(lm), lm)], threads)

    def load(self, label_member, scale=0.5):
        """RGB image and label map (both resized by `scale`)."""
        pi, pl = self.z.get([self._image_member(label_member), label_member])
        img, lab = _rgb(pi), cv2.imread(pl, cv2.IMREAD_UNCHANGED)
        if scale != 1:
            w, h = int(img.shape[1] * scale), int(img.shape[0] * scale)
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
            lab = cv2.resize(lab, (w, h), interpolation=cv2.INTER_NEAREST)
        return img, lab

    def traversability(self, lab):
        """0 = ignored (sky, own vehicle), 1 = traversable, 2 = conditionally traversable, 3 = obstacle."""
        lut = np.full(256, 3, np.uint8)
        for group, v in ((self.TRAVERSABLE, 1), (self.CONDITIONAL, 2), (self.IGNORE, 0)):
            for n in group:
                if n in self.id_of:
                    lut[self.id_of[n]] = v
        return lut[lab]


# =============================================================================
# 3. Camera geometry
# =============================================================================
def gsd_nadir(altitude, hfov_deg, width_px):
    """Ground sampling distance of a vertical view, m/px: GSD = 2 H tan(alpha/2) / N."""
    return 2 * altitude * math.tan(math.radians(hfov_deg) / 2) / width_px


def hfov_from_sensor(focal_mm, pixel_um, width_px):
    """Horizontal field of view, deg, from the focal length and the pixel pitch."""
    return math.degrees(2 * math.atan(width_px * pixel_um * 1e-3 / 2 / focal_mm))


# =============================================================================
# 4. Degradations of the video signal (applied to real frames)
# =============================================================================
def fractal_noise(shape, rng, octaves=5, start=3, persistence=0.5):
    """Smooth random field in [0, 1] (sum of up-sampled random grids)."""
    h, w = shape
    out = np.zeros((h, w), np.float32)
    amp, m = 1.0, max(h, w)
    for o in range(octaves):
        n = start * (2 ** o)
        grid = rng.random((max(2, int(round(n * h / m)) + 1), max(2, int(round(n * w / m)) + 1)), dtype=np.float32)
        out += amp * cv2.resize(grid, (w, h), interpolation=cv2.INTER_CUBIC)
        amp *= persistence
    out -= out.min()
    return out / max(float(out.max()), 1e-6)


def _f(img):
    return img.astype(np.float32)


def _u8(x):
    return np.clip(x, 0, 255).astype(np.uint8)


def motion_blur(img, length=9.0, angle=0.0):
    """Linear motion blur: the point spread function is a line of `length` px at `angle` deg."""
    if length < 1.5:
        return img.copy()
    size = int(math.ceil(length)) | 1
    k = np.zeros((size * 4, size * 4), np.uint8)
    c = size * 2
    dx = math.cos(math.radians(angle)) * (length - 1) * 2
    dy = -math.sin(math.radians(angle)) * (length - 1) * 2
    cv2.line(k, (int(c - dx), int(c - dy)), (int(c + dx), int(c + dy)), 255, 4, cv2.LINE_AA)
    k = cv2.resize(k.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
    return cv2.filter2D(img, -1, k / max(k.sum(), 1e-6), borderType=cv2.BORDER_REFLECT)


def defocus(img, sigma=1.5):
    return img.copy() if sigma <= 0 else cv2.GaussianBlur(img, (0, 0), sigma)


def add_noise(img, sigma=10.0, seed=None):
    rng = np.random.default_rng(seed)
    return _u8(_f(img) + rng.normal(0, sigma, img.shape))


def low_light(img, level=0.15, gain_max=8.0, seed=None):
    """Dark scene and automatic gain: light reduced by `level`, photon and read noise added."""
    if level >= 0.999:
        return img.copy()
    rng = np.random.default_rng(seed)
    lin = (_f(img) / 255.0) ** 2.2 * level
    full_well = 3000.0
    noisy = lin + rng.standard_normal(lin.shape).astype(np.float32) * np.sqrt(lin / full_well + (4.0 / full_well) ** 2)
    gain = float(np.clip(0.16 / max(float(lin.mean()), 1e-6), 1.0, gain_max))
    return _u8(np.clip(noisy * gain, 0, 1) ** (1 / 2.2) * 255)


def haze(img, density=0.4, airlight=200.0):
    """Uniform haze: I = J t + A (1 - t), transmission t = 1 - density."""
    t = 1.0 - float(np.clip(density, 0, 0.98))
    return _u8(_f(img) * t + airlight * (1 - t))


def smoke(img, density=0.5, seed=0, color=170.0, opacity=0.92):
    """Smoke: spatially variable veil."""
    if density <= 0:
        return img.copy()
    n = fractal_noise(img.shape[:2], np.random.default_rng(seed))
    m = np.clip((n - (1.0 - density)) * 3.0, 0, 1) * opacity
    if img.ndim == 3:
        m = m[..., None]
    return _u8(_f(img) * (1 - m) + color * m)


def jpeg(img, quality=30):
    """JPEG compression artefacts."""
    if quality >= 100:
        return img.copy()
    x = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode(".jpg", x, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    y = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    return y if img.ndim == 2 else cv2.cvtColor(y, cv2.COLOR_BGR2RGB)


def downscale(img, factor=0.5):
    """Loss of resolution: down-sample by `factor`, then up-sample back."""
    if factor >= 0.999:
        return img.copy()
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def rolling_shutter(img, amplitude=6.0, cycles=1.5, phase=0.0):
    """Rolling-shutter distortion: every row is shifted by a sinusoid of the row number."""
    if amplitude <= 0:
        return img.copy()
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    shift = amplitude * np.sin(2 * np.pi * cycles * yy / h + phase)
    return cv2.remap(img, xx + shift, yy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)


def analog_interference(img, strength=0.5, seed=None):
    """Analog FPV video link with a weak or jammed signal: colour loss, snow, line
    noise, horizontal tearing (loss of synchronisation) and a rolling brightness bar."""
    if strength <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    h = img.shape[0]
    x = _f(img)
    if x.ndim == 3:
        gray = x.mean(axis=2, keepdims=True)
        x = gray + (x - gray) * (1 - 0.85 * min(strength * 1.3, 1))
    x += rng.normal(0, 45 * strength, x.shape)
    for r in rng.integers(0, h, int(h * 0.06 * strength)):
        x[r] += rng.normal(70, 40) * strength
    if strength > 0.25:
        y0, band = rng.integers(0, h), int(h * 0.25 * strength)
        offs = np.cumsum(rng.normal(0, 3.0 * strength, band))
        for i in range(band):
            r = (y0 + i) % h
            x[r] = np.roll(x[r], int(offs[i] + 30 * strength), axis=0)
    bar = 1.0 + 0.3 * strength * np.sin(2 * np.pi * (np.arange(h, dtype=np.float32)[:, None] / h * 1.3 + rng.random()))
    return _u8(x * (bar if x.ndim == 2 else bar[..., None]))


def packet_loss(img, rate=0.05, block=16, seed=None):
    """Digital link with lost packets: damaged macroblock slices are filled with a smear
    of the row above or with a grey block."""
    if rate <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    out = img.copy()
    gh, gw = img.shape[0] // block, img.shape[1] // block
    lost = rng.random((gh, gw)) < rate
    for r in range(gh):
        c = 0
        while c < gw:
            if lost[r, c]:
                run = int(rng.integers(1, max(2, gw // 3)))
                lost[r, c:c + run] = True
                c += run
            c += 1
    for r, c in zip(*np.nonzero(lost)):
        ys, xs = slice(r * block, (r + 1) * block), slice(c * block, (c + 1) * block)
        if r > 0 and rng.random() < 0.6:
            out[ys, xs] = out[r * block - 1:r * block, xs]
        else:
            out[ys, xs] = 128 if img.ndim == 2 else (60, 128, 60)
    return out


def degrade(img, sensor="visible", motion_px=0.0, motion_angle=0.0, defocus_sigma=0.0, light=1.0, noise=0.0,
            haze_density=0.0, smoke_density=0.0, rolling=0.0, scale=1.0, jpeg_quality=100, analog=0.0,
            packet=0.0, seed=0):
    """Chain of degradations in physical order: atmosphere -> optics and motion -> sensor ->
    resolution -> compression -> radio link. For the thermal sensor haze and smoke are
    attenuated (long-wave infrared passes thin smoke and haze) and low light does not apply."""
    x, ir = img, sensor == "thermal"
    if haze_density > 0:
        x = haze(x, haze_density * (0.25 if ir else 1.0), airlight=128 if ir else 200)
    if smoke_density > 0:
        x = smoke(x, smoke_density, seed=seed, color=128 if ir else 170, opacity=0.18 if ir else 0.92)
    if motion_px > 0:
        x = motion_blur(x, motion_px, motion_angle)
    if defocus_sigma > 0:
        x = defocus(x, defocus_sigma)
    if rolling > 0:
        x = rolling_shutter(x, rolling)
    if light < 1.0 and not ir:
        x = low_light(x, light, seed=seed)
    if noise > 0:
        x = add_noise(x, noise, seed=seed + 1)
    if scale < 1.0:
        x = downscale(x, scale)
    if jpeg_quality < 100:
        x = jpeg(x, jpeg_quality)
    if analog > 0:
        x = analog_interference(x, analog, seed=seed + 2)
    if packet > 0:
        x = packet_loss(x, packet, seed=seed + 3)
    return x


# =============================================================================
# 5. Measures, classical detection, tracking
# =============================================================================
def gray(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


def psnr(ref, test):
    """Peak signal-to-noise ratio, dB."""
    mse = float(np.mean((_f(ref) - _f(test)) ** 2))
    return float("inf") if mse == 0 else 10 * math.log10(255.0 ** 2 / mse)


def ssim(ref, test):
    """Structural similarity index of grey levels (Gaussian window 11 x 11, sigma 1.5)."""
    a, b = _f(gray(ref)), _f(gray(test))
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    blur = lambda z: cv2.GaussianBlur(z, (11, 11), 1.5)
    ma, mb = blur(a), blur(b)
    va, vb, cab = blur(a * a) - ma * ma, blur(b * b) - mb * mb, blur(a * b) - ma * mb
    return float((((2 * ma * mb + c1) * (2 * cab + c2)) / ((ma * ma + mb * mb + c1) * (va + vb + c2))).mean())


def sharpness(img):
    """Variance of the Laplacian: a measure of focus (small values = blurred image)."""
    return float(cv2.Laplacian(gray(img), cv2.CV_64F).var())


def to_white_hot(img, palette):
    """Thermal frame in the white-hot convention (warm = bright) as a 3-channel uint8 image.
    palette: "white-hot" (HIT-UAV), "black-hot" (M3OT, inverted here) or a pseudo-colour palette
    whose luminance grows with the temperature (Kust4K: purple = cold, yellow = warm)."""
    g = gray(img)
    if palette.startswith("black"):
        g = 255 - g
    return np.dstack([g, g, g])


def vegetation_mask(rgb, exg_min=0.12, v_min=35, erode_px=15):
    """Vegetation of a visible RGB frame by the excess-green index ExG = (2G - R - B) / (R + G + B).
    The mask is eroded by erode_px so that a residual offset between the visible and the thermal
    channel does not move it onto neighbouring surfaces."""
    f = rgb.astype(np.float32)
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    exg = (2 * g - r - b) / (r + g + b + 1e-6)
    m = ((exg > exg_min) & (f.max(axis=2) > v_min)).astype(np.uint8)
    if erode_px > 1:
        m = cv2.erode(m, np.ones((erode_px, erode_px), np.uint8))
    return m.astype(bool)


def surroundings_mask(shape, boxes, scale=1.0, gap=3):
    """Background ring around annotated objects (e.g. the road surface around vehicles): every box
    [x1, y1, x2, y2] is expanded by scale x its longer side; the boxes themselves, expanded by gap px,
    are excluded."""
    ring = np.zeros(shape[:2], np.uint8)
    core = np.zeros(shape[:2], np.uint8)
    for x1, y1, x2, y2 in boxes:
        s = int(round(scale * max(x2 - x1, y2 - y1)))
        cv2.rectangle(ring, (int(x1) - s, int(y1) - s), (int(x2) + s, int(y2) + s), 1, -1)
        cv2.rectangle(core, (int(x1) - gap, int(y1) - gap), (int(x2) + gap, int(y2) + gap), 1, -1)
    return (ring > 0) & (core == 0)


def palette_check(thermal, road, vegetation, margin=15, min_pixels=200):
    """Palette of a thermal frame established from the scene. In a road scene the asphalt is, as a
    rule, warmer than vegetation (by day it absorbs most of the solar radiation, at night it gives off
    the stored heat), therefore a road darker than vegetation indicates the black-hot palette
    (warm = dark) and a brighter road the white-hot convention (warm = bright). A shaded or wet road
    and sunlit tree crowns can reverse the order in single frames: check several frames."""
    g = gray(thermal)
    road = road & ~vegetation
    nr, nv = int(road.sum()), int(vegetation.sum())
    r = float(np.median(g[road])) if nr >= min_pixels else float("nan")
    v = float(np.median(g[vegetation])) if nv >= min_pixels else float("nan")
    if np.isnan(r) or np.isnan(v):
        verdict = "not determined: too few road or vegetation pixels"
    elif r < v - margin:
        verdict = "road darker than vegetation: black-hot (warm = dark)"
    elif r > v + margin:
        verdict = "road brighter than vegetation: white-hot (warm = bright)"
    else:
        verdict = f"difference below {margin} grey levels: not determined"
    return dict(road_median=r, vegetation_median=v, road_pixels=nr, vegetation_pixels=nv, verdict=verdict)


def signal_to_clutter(img, mask, exclude=None, gap=2, ring=8):
    """Signal-to-clutter ratio of an object: |mean of object - mean of background ring| /
    standard deviation of the background ring (grey levels)."""
    g = _f(gray(img))
    m = mask.astype(np.uint8)
    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * gap + 1, 2 * gap + 1))
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * (gap + ring) + 1, 2 * (gap + ring) + 1))
    bg = (cv2.dilate(m, k2) > 0) & (cv2.dilate(m, k1) == 0)
    if exclude is not None:
        bg &= ~exclude
    t, b = g[m > 0], g[bg]
    if t.size < 3 or b.size < 10:
        return float("nan")
    return abs(float(t.mean()) - float(b.mean())) / max(float(b.std()), 1.0)


def box_iou(a, b):
    """Intersection over union of two boxes [x1, y1, x2, y2]."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def boxes_from_labels(labels, class_ids, names, min_area=20):
    """Boxes of connected regions of the given classes in a label map."""
    out = []
    for c in class_ids:
        n, cc, stats, _ = cv2.connectedComponentsWithStats((labels == c).astype(np.uint8), connectivity=8)
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a >= min_area:
                out.append(dict(cls=names[c], class_id=int(c), box=[float(x), float(y), float(x + w), float(y + h)],
                                area=int(a)))
    return out


def patch_labels(labels, tile, object_ids, names, min_px=10):
    """Patch classification from a label map: every tile receives the object class with
    the largest number of pixels in it, or 'background'."""
    rows, cols = labels.shape[0] // tile, labels.shape[1] // tile
    grid = np.empty((rows, cols), object)
    for r in range(rows):
        for c in range(cols):
            t = labels[r * tile:(r + 1) * tile, c * tile:(c + 1) * tile]
            n, k = max((int((t == k).sum()), k) for k in object_ids)
            grid[r, c] = names[k] if n >= min_px else "background"
    return grid


def detect_hotspots(img, k=5.0, se_px=21, min_area=4, max_area=5000):
    """Classical detector of small warm objects in a white-hot thermal image: white
    top-hat (local contrast) -> threshold mean + k std -> connected components.
    Returns boxes (N, 4) [x1, y1, x2, y2] and the binary mask."""
    g = cv2.GaussianBlur(gray(img), (3, 3), 0)
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(se_px) | 1, int(se_px) | 1))
    th = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, se).astype(np.float32)
    mask = (th > th.mean() + k * th.std()).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    boxes = [[x, y, x + w, y + h] for x, y, w, h, a in stats[1:] if min_area <= a <= max_area]
    return np.array(boxes, float).reshape(-1, 4), mask


def expected_size_px(gsd, size_m=0.45, margin_px=6.0):
    """Expected size of an object in an annotated box, px: size_m / GSD + margin_px.
    Defaults: nominal shoulder width of a person 0.45 m and a margin of 6 px (blur, thermal halo,
    annotation margin). For comparison, a least-squares fit to the HIT-UAV boxes of persons in
    vertical views gives d = 0.38 m and m = 6.6 px; block 1.1.1-B computes the fit for every class."""
    return size_m / gsd + margin_px


def detect_warm_objects(img, gsd, k=6.0, size_m=0.45, margin_px=6.0):
    """Hot-spot detector adapted to the ground sampling distance: the structuring element
    and the admissible blob areas follow the expected size of a person in pixels."""
    s = expected_size_px(gsd, size_m, margin_px)
    return detect_hotspots(img, k=k, se_px=int(max(7, round(2.5 * s))) | 1,
                           min_area=max(3, int((0.3 * s) ** 2)), max_area=int((2.0 * s) ** 2))


def match_detections(pred, objects, target_classes, pad=2.0):
    """A detection is correct when its centre lies inside a (padded) box of an object of a
    target class. Detections inside boxes of other annotated objects are counted apart.
    Returns dict(tp, fp, other, missed, recall, precision)."""
    tgt = [o for o in objects if o["cls"] in target_classes]
    oth = [o for o in objects if o["cls"] not in target_classes]
    used, tp, fp, other = set(), [], [], []
    inside = lambda o, cx, cy: (o["box"][0] - pad <= cx <= o["box"][2] + pad and o["box"][1] - pad <= cy <= o["box"][3] + pad)
    for b in pred:
        cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
        hit = next((i for i, o in enumerate(tgt) if i not in used and inside(o, cx, cy)), None)
        if hit is not None:
            used.add(hit)
            tp.append(list(b))
        elif any(inside(o, cx, cy) for o in oth) or any(inside(o, cx, cy) for o in tgt):
            other.append(list(b))
        else:
            fp.append(list(b))
    n = len(tgt)
    return dict(tp=tp, fp=fp, other=other, missed=[o for i, o in enumerate(tgt) if i not in used],
                recall=len(tp) / n if n else float("nan"), precision=len(tp) / max(len(tp) + len(fp), 1))


def global_shift(prev, cur):
    """Translation of the image content from `prev` to `cur` (phase correlation), px."""
    (dx, dy), _ = cv2.phaseCorrelate(_f(gray(prev)), _f(gray(cur)))
    return dx, dy


class NearestNeighbourTracker:
    """Simple multi-object tracker: the detections of a frame are linked to the tracks of
    the previous frame by the smallest distance after compensation of the camera motion.
    A track is closed after `max_missed` frames without a detection."""

    def __init__(self, max_dist=12.0, max_missed=4):
        self.max_dist, self.max_missed = max_dist, max_missed
        self.tracks, self.closed, self.next_id = {}, {}, 1
        self.offset = np.zeros(2)                    # accumulated camera motion, px

    def update(self, centres, shift=(0.0, 0.0), frame=0):
        """centres: (N, 2) detections of the frame; shift: image motion since the previous frame.
        Returns the list of (track id, centre) of the frame."""
        self.offset += np.asarray(shift, float)
        pts = np.asarray(centres, float).reshape(-1, 2) - self.offset      # coordinates of the first frame
        ids = list(self.tracks)
        pairs = []
        for i, tid in enumerate(ids):
            last = self.tracks[tid]["hist"][-1][1]
            for j, p in enumerate(pts):
                d = float(np.hypot(*(p - last)))
                if d <= self.max_dist:
                    pairs.append((d, tid, j))
        pairs.sort()
        used_t, used_d, out = set(), set(), []
        for d, tid, j in pairs:
            if tid in used_t or j in used_d:
                continue
            used_t.add(tid)
            used_d.add(j)
            self.tracks[tid]["hist"].append((frame, pts[j]))
            self.tracks[tid]["missed"] = 0
            out.append((tid, pts[j] + self.offset))
        for tid in ids:
            if tid not in used_t:
                self.tracks[tid]["missed"] += 1
                if self.tracks[tid]["missed"] > self.max_missed:
                    self.closed[tid] = self.tracks.pop(tid)
        for j, p in enumerate(pts):
            if j not in used_d:
                self.tracks[self.next_id] = dict(hist=[(frame, p)], missed=0)
                out.append((self.next_id, p + self.offset))
                self.next_id += 1
        return out

    def history(self, tid):
        """Trajectory of a track in the coordinates of the current frame."""
        t = self.tracks.get(tid) or self.closed.get(tid)
        return np.array([p for _, p in t["hist"]]) + self.offset


def track_warm_objects(frames, gsd, fps, k=5.0, max_speed_ms=2.5, max_missed=4, min_len=12, min_disp_m=2.0):
    """Detection and tracking of warm objects in a thermal video:
    detect_warm_objects -> camera motion by phase correlation -> NearestNeighbourTracker.
    A track is reported as a moving object when it lasts at least `min_len` frames and its
    start and end points are at least `min_disp_m` apart on the ground.
    Returns (tracker, moving: {id: trajectory (K, 2) in the last frame}, rows: list of
    (frame, track id, x, y), detections per frame)."""
    tr = NearestNeighbourTracker(max_dist=max(5.0, max_speed_ms / gsd / max(fps, 1e-6) + 3.0), max_missed=max_missed)
    prev, rows, n_det = None, [], []
    for i, f in enumerate(frames):
        g = gray(f)
        b, _ = detect_warm_objects(g, gsd, k=k)
        c = np.c_[(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2] if len(b) else np.zeros((0, 2))
        shift = global_shift(prev, g) if prev is not None else (0.0, 0.0)
        for tid, p in tr.update(c, shift, i):
            rows.append((i, tid, float(p[0]), float(p[1])))
        n_det.append(len(b))
        prev = g
    moving = {}
    for tid, t in {**tr.closed, **tr.tracks}.items():
        pts = np.array([p for _, p in t["hist"]])
        if len(pts) >= min_len and float(np.hypot(*(pts[-1] - pts[0]))) * gsd >= min_disp_m:
            moving[tid] = pts + tr.offset
    return tr, moving, rows, n_det


def video_info(path):
    """Properties of a video file read by OpenCV."""
    cap = cv2.VideoCapture(path)
    info = dict(path=path, opened=cap.isOpened(), fps=cap.get(cv2.CAP_PROP_FPS),
                frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                codec=int(cap.get(cv2.CAP_PROP_FOURCC)).to_bytes(4, "little").decode("latin-1", "replace"))
    cap.release()
    info["duration_s"] = info["frames"] / info["fps"] if info["fps"] else float("nan")
    info["size_mb"] = os.path.getsize(path) / 1e6 if os.path.exists(path) else float("nan")
    return info


def iter_video(path, every_n=1, max_frames=None, start=0):
    """Generator of (frame number, timestamp s, RGB frame) of every n-th frame of a video:
    frames are decoded one by one, so a long video does not fill the memory."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    k, n_out = start, 0
    try:
        while True:
            ok, f = cap.read()
            if not ok or (max_frames is not None and n_out >= max_frames):
                break
            if (k - start) % every_n == 0:
                yield k, k / fps, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
                n_out += 1
            k += 1
    finally:
        cap.release()


def write_video(path, frames, fps, codec="mp4v"):
    """Write RGB or grey frames to a video file; returns the path. Codec 'mp4v' (.mp4) works in
    Colab and in most OpenCV builds; 'MJPG' (.avi) stores every frame as JPEG."""
    h, w = frames[0].shape[:2]
    wr = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*codec), float(fps), (w, h))
    if not wr.isOpened():
        raise IOError(f"OpenCV cannot write the codec {codec}")
    for f in frames:
        f = to_rgb(f)
        if f.shape[:2] != (h, w):
            f = cv2.resize(f, (w, h))
        wr.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    wr.release()
    return path


def contact_sheet(frames, cols=6, width=1600, labels=None, font=0.5):
    """Grid of frames for a quick visual review of a sequence."""
    n = len(frames)
    rows = int(math.ceil(n / cols))
    tw = width // cols
    th = int(round(tw * frames[0].shape[0] / frames[0].shape[1]))
    sheet = np.zeros((rows * th, cols * tw, 3), np.uint8)
    for i, f in enumerate(frames):
        r, c = divmod(i, cols)
        t = cv2.resize(to_rgb(f), (tw, th), interpolation=cv2.INTER_AREA)
        if labels is not None:
            cv2.rectangle(t, (0, 0), (tw, 18), (0, 0, 0), -1)
            cv2.putText(t, str(labels[i]), (4, 13), cv2.FONT_HERSHEY_SIMPLEX, font * 0.8, (255, 255, 0), 1, cv2.LINE_AA)
        sheet[r * th:(r + 1) * th, c * tw:(c + 1) * tw] = t
    return sheet


def frame_difference(a, b):
    """Mean absolute difference of the grey levels of two frames."""
    return float(np.mean(np.abs(_f(gray(a)) - _f(gray(b)))))


def grad_magnitude(img, sigma=1.5):
    """Normalised gradient magnitude; edges of a visible and a thermal image of the same scene
    are similar even when their brightness is not, so it is used to compare the two channels."""
    g = cv2.GaussianBlur(_f(gray(img)), (0, 0), sigma)
    m = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    return (m - m.mean()) / (m.std() + 1e-6)


def channel_shift(img_a, img_b):
    """Translation between two images of different sensors (phase correlation of gradient
    magnitudes): (dx, dy, response). The response is low when the images have little common structure."""
    a, b = grad_magnitude(img_a), grad_magnitude(img_b)
    win = cv2.createHanningWindow((a.shape[1], a.shape[0]), cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(a, b, win)
    return float(dx), float(dy), float(resp)


def track_offsets(table_a, table_b, lag=0):
    """Offsets of the centres of the same track (same id) in two annotated channels:
    channel b at frame t is compared with channel a at frame t + lag. Returns arrays dx, dy."""
    bb = table_b[["id", "frame", "cx", "cy"]].copy()
    bb["frame"] = bb["frame"] + lag
    m = bb.merge(table_a[["id", "frame", "cx", "cy"]], on=["id", "frame"], suffixes=("_b", "_a"))
    return (m.cx_b - m.cx_a).values, (m.cy_b - m.cy_a).values


def read_video(path, step=1, max_frames=None, start=0):
    """Frames of a video file (BGR from OpenCV converted to RGB) and the frame rate."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames, k = [], 0
    while True:
        ok, f = cap.read()
        if not ok or (max_frames and len(frames) >= max_frames):
            break
        if k % step == 0:
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        k += 1
    cap.release()
    return frames, fps


# =============================================================================
# 6. Visualisation
# =============================================================================
def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else img


def colorize(labels, colors):
    """Label map -> RGB image with the colour table `colors` (array K x 3)."""
    return np.asarray(colors, np.uint8)[np.clip(labels, 0, len(colors) - 1)]


def overlay(img, color_img, alpha=0.5, mask=None):
    base, col = _f(to_rgb(img)), _f(color_img)
    a = np.full(base.shape[:2], alpha, np.float32) if mask is None else mask.astype(np.float32) * alpha
    return _u8(base * (1 - a[..., None]) + col * a[..., None])


def draw_boxes(img, objects, colors=None, labels=True, scores=None, ids=None, thickness=1, font=0.4, color=None):
    """Draw boxes of objects (dicts with 'box' and 'cls')."""
    out = to_rgb(img).copy()
    for i, o in enumerate(objects):
        x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
        c = color if color is not None else (colors or {}).get(o["cls"], (255, 255, 0))
        c = tuple(int(v) for v in c)
        cv2.rectangle(out, (x1, y1), (x2, y2), c, thickness, cv2.LINE_AA)
        txt = []
        if labels:
            txt.append(str(o["cls"]))
        if ids is not None:
            txt.append(f"#{ids[i]}")
        if scores is not None:
            txt.append(f"{scores[i]:.2f}")
        if txt:
            s = " ".join(txt)
            (tw, th), _ = cv2.getTextSize(s, cv2.FONT_HERSHEY_SIMPLEX, font, 1)
            y0 = max(y1 - 3, th + 2)
            cv2.rectangle(out, (x1, y0 - th - 2), (x1 + tw + 2, y0 + 2), c, -1)
            cv2.putText(out, s, (x1 + 1, y0), cv2.FONT_HERSHEY_SIMPLEX, font, (0, 0, 0), 1, cv2.LINE_AA)
    return out


def legend(ax, names, colors, loc="lower right", fontsize=7, ncol=1):
    import matplotlib.patches as mpatches
    ax.legend(handles=[mpatches.Patch(color=np.array(c) / 255.0, label=n) for n, c in zip(names, colors)],
              loc=loc, fontsize=fontsize, ncol=ncol, framealpha=0.85)


def setup_plots():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8,
                         "ytick.labelsize": 8, "legend.fontsize": 8, "figure.facecolor": "white"})


def catalogue():
    """Table of the data sets of the course."""
    return [
        ("HIT-UAV", "UAV DJI M210 V2, thermal camera Zenmuse XT2 (640x512, 25 x 20 deg)",
         "2898 thermal images, 5 thermal videos; boxes of persons and vehicles; 60-130 m, day and night",
         "CC BY 4.0", "Suo J. et al., Scientific Data 10, 227 (2023)",
         "https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset"),
        ("Kust4K", "UAV hexacopter HET-3021C, visible + thermal cameras",
         "4024 pixel-aligned RGB-thermal pairs 640x512 over urban roads, day and night; pixel labels of 8 classes",
         "CC BY 4.0", "Scientific Data 12, 1701 (2025)", "https://doi.org/10.6084/m9.figshare.29476610"),
        ("VisDrone2019-DET (val)", "camera drones of different types, 14 Chinese cities",
         "548 RGB images; boxes of pedestrians, cars, vans, trucks, buses, motors and others",
         "research use, see the data set site", "Zhu P. et al., IEEE TPAMI 44(11), 7380-7399 (2022)",
         "https://github.com/VisDrone/VisDrone-Dataset"),
        ("GOOSE (val)", "ground vehicle, front RGB camera 2048x1000",
         "962 images of roads, field tracks, forests and a training area; pixel labels of 64 classes",
         "CC BY-SA 4.0", "IEEE ICRA 2024 (arXiv:2310.16788)", "https://goose-dataset.de"),
    ]
'''
WORK_DIR = "/content" if IN_COLAB else os.getcwd()
with open(os.path.join(WORK_DIR, "aitbc.py"), "w", encoding="utf-8") as fh:
    fh.write(LIB_SOURCE)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
import aitbc
aitbc = importlib.reload(aitbc)

if os.environ.get("AITBC_DATA"):
    cache_dir = os.environ["AITBC_DATA"]
elif CACHE_ON_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    cache_dir = "/content/drive/MyDrive/AITBC/data_cache"
else:
    cache_dir = os.path.join(WORK_DIR, "aitbc_data")
aitbc.set_data_root(cache_dir)

t0 = time.time()
hit, vis, m3ot = aitbc.HITUAV(), aitbc.VisDrone(), aitbc.M3OT()
ENVIRONMENT["aitbc"] = aitbc.__version__
n_files, size_mb = aitbc.cache_size()
print(f"aitbc {aitbc.__version__}: HIT-UAV ({len(hit.images)} thermal images, {len(hit.objects)} objects), "
      f"VisDrone ({len(vis.names())} images), M3OT ({len(m3ot.sequences())} sequences) connected in {time.time() - t0:.0f} s")
print(f"Cache {aitbc.DATA_ROOT}: {n_files} files, {size_mb:.1f} MB")

---
## Topic 1. Google Colab and video data

A UAV camera produces a video stream: a sequence of frames with a fixed frame rate $f_v$ (frames per second). For analysis the stream is usually **decomposed into frames** with a lower sampling rate $f_s \le f_v$: neighbouring frames of a 10–30 frames/s video are almost identical, and processing every frame costs time without new information.

### 1.1. Real UAV video fragment

**Purpose.** Build a short video file from a real image sequence of the M3OT data set (DJI Mavic 3T, 100–120 m, 10 frames/s) and check its properties with OpenCV.

**Parameters.**
- `SEQUENCE` — scene of the flight (day, dusk, night; road, crossroads, highway).
- `CHANNEL` — visible camera or thermal camera of the same UAV.
- `START_FRAME`, `N_FRAMES` — first frame and length of the fragment. One frame is about 0.6 MB, a fragment of 60 frames (6 s) is about 35 MB.
- `CODEC` — `mp4v` (.mp4, compact) or `MJPG` (.avi, every frame is a JPEG image).

**What to look at.** The table of video properties (frame rate, number of frames, duration, size); the compression ratio of the video against the source PNG frames; the frames with annotated vehicles: the boxes move between the frames, the scene stays in place.

In [ ]:
#@title 1.1 Real UAV video fragment (M3OT) { display-mode: "form" }
SEQUENCE = "1-04 · day, suburban highway" #@param ["1-01 · day, suburban road", "1-02 · night, city crossroads", "1-03 · day, city road", "1-04 · day, suburban highway", "1-05 · day, crossroads", "1-06 · dusk, highway", "1-07 · day, crossroads", "1-08 · dusk, crossroads", "1-09 · day, city street with parked cars"]
CHANNEL = "visible" #@param ["visible", "thermal"]
START_FRAME = 1 #@param {type:"slider", min:1, max:240, step:1}
N_FRAMES = 60 #@param {type:"slider", min:20, max:150, step:10}
CODEC = "mp4v (.mp4)" #@param ["mp4v (.mp4)", "MJPG (.avi)"]

SEQ = SEQUENCE.split(" ")[0]
CH = "rgb" if CHANNEL == "visible" else "ir"
FPS = m3ot.CAMERA["fps"]
n_total = m3ot.n_frames(SEQ, CH)
start = int(min(START_FRAME - 1, max(n_total - N_FRAMES, 0)))
count = int(min(N_FRAMES, n_total - start))

t0 = time.time()
png_paths = m3ot.prefetch(SEQ, CH, start=start, count=count)
FRAMES = m3ot.frames(SEQ, CH, start=start, count=count)
t_download = time.time() - t0
fourcc, ext = ("mp4v", ".mp4") if CODEC.startswith("mp4v") else ("MJPG", ".avi")
VIDEO = os.path.join(DIRS["data"], f"m3ot_{SEQ}_{CH}_{start + 1:04d}_{count}{ext}")
aitbc.write_video(VIDEO, FRAMES, fps=FPS, codec=fourcc)
info = aitbc.video_info(VIDEO)
png_mb = sum(os.path.getsize(p) for p in png_paths) / 1e6
TRACKS = m3ot.tracks(SEQ, CH)                               # annotation of the whole sequence

table = pd.DataFrame([
    ("Sequence", f"{SEQ} ({m3ot.SEQUENCES[SEQ]}), {CHANNEL} channel"),
    ("Frames of the fragment", f"{start + 1}–{start + count} of {n_total}"),
    ("Frame size", f"{info['width']} × {info['height']} px"),
    ("Frame rate f_v", f"{info['fps']:.1f} frames/s"),
    ("Duration", f"{info['duration_s']:.1f} s"),
    ("Codec", info["codec"]),
    ("Source PNG frames", f"{png_mb:.1f} MB"),
    ("Video file", f"{info['size_mb']:.2f} MB (compression ratio {png_mb / max(info['size_mb'], 1e-6):.0f} : 1)"),
    ("Annotated vehicle tracks in the sequence", int(TRACKS.id.nunique())),
    ("Time of download and assembly", f"{t_download:.0f} s"),
], columns=["Property", "Value"]).set_index("Property")
display(table)

picks = [0, count // 2, count - 1]
views, titles = [], []
for k in picks:
    objs = m3ot.objects(TRACKS, start + k + 1)
    views.append(aitbc.draw_boxes(FRAMES[k], objs, color=(255, 230, 0), labels=False, thickness=2))
    titles.append(f"frame {start + k + 1}, t = {k / FPS:.1f} s, {len(objs)} vehicles")
show_images(views, titles, "1_1_video_frames.png", cols=3, width=4.6)

### 1.2. Decomposition of the video into frames

**Purpose.** Write a function that reads the video and saves frames with a given sampling rate $f_s$, and build the table of time stamps. A frame with number $n$ has the time stamp $t_n = n / f_v$; a frame is saved when $t_n$ reaches the next sampling instant $0,\ 1/f_s,\ 2/f_s, \dots$ The time stamp of the last frame is $(N_v - 1)/f_v$, where $N_v$ is the number of frames in the video, therefore the number of saved frames is

$$N_s = \min\left(\left\lfloor \frac{(N_v - 1)\, f_s}{f_v} \right\rfloor + 1,\; N_v\right).$$

When $f_s \ge f_v$ every frame is saved and a further increase of $f_s$ gives nothing.

**Parameters.**
- `SAMPLING_FPS` — sampling rate $f_s$, frames/s.
- `MAX_FRAMES` — upper limit of the number of saved frames.

**What to look at.**
1. The **motion-activity plot**: the mean absolute difference of neighbouring frames shows when the scene changes (moving vehicles, camera motion). The saved frames are marked on it: with a low $f_s$ short events fall between the samples.
2. The plot $N_s(f_s)$ with the saturation at $f_s = f_v$.
3. The contact sheet: a quick visual review of the saved frames.

In [ ]:
#@title 1.2 Decomposition of the video into frames { display-mode: "form" }
SAMPLING_FPS = 2.0 #@param {type:"slider", min:0.5, max:12, step:0.5}
MAX_FRAMES = 100 #@param {type:"slider", min:5, max:300, step:5}


def extract_frames(video_path, destination, sampling_fps=2.0, max_frames=100):
    # save frames of a video with the sampling rate sampling_fps; returns the table of time stamps
    if sampling_fps <= 0 or max_frames < 1:
        raise ValueError("The sampling rate must be positive and the number of frames at least one.")
    os.makedirs(destination, exist_ok=True)
    for old in os.listdir(destination):
        if old.startswith("frame_") and old.endswith(".jpg"):
            os.remove(os.path.join(destination, old))
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise OSError(f"Cannot open the video file {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        cap.release()
        raise ValueError("The video file has no valid frame rate.")
    records, index, next_sample = [], 0, 0.0
    try:
        while len(records) < max_frames:
            ok, frame = cap.read()
            if not ok:
                break
            timestamp = index / fps
            if timestamp + 1e-9 >= next_sample:
                name = f"frame_{len(records):04d}.jpg"
                if not cv2.imwrite(os.path.join(destination, name), frame):
                    raise OSError(f"Cannot save the frame {name}")
                records.append({"frame": name, "time_s": round(timestamp, 3), "source_index": index})
                next_sample += 1.0 / sampling_fps
            index += 1
    finally:
        cap.release()
    return pd.DataFrame(records)


frames_table = extract_frames(VIDEO, DIRS["frames"], SAMPLING_FPS, MAX_FRAMES)
frames_table.to_csv(os.path.join(OUT, "frame_times.csv"), index=False)
interval = frames_table.time_s.diff().dropna()
print(f"Saved {len(frames_table)} frames of {info['frames']}; interval between samples "
      f"{interval.mean():.2f} s (requested {1 / SAMPLING_FPS:.2f} s)")
display(frames_table.head(8))

# motion activity: mean absolute difference of neighbouring frames of the whole fragment
activity = np.array([aitbc.frame_difference(FRAMES[i - 1], FRAMES[i]) for i in range(1, count)])
t_act = np.arange(1, count) / FPS
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.7, 1]})
axes[0].plot(t_act, activity, color=COLORS[0], lw=1.4, label="difference of neighbouring frames")
for t in frames_table.time_s:
    axes[0].axvline(t, color=COLORS[1], lw=0.8, alpha=0.6)
axes[0].plot([], [], color=COLORS[1], lw=0.8, label=f"saved frames, f_s = {SAMPLING_FPS} frames/s")
axes[0].set(xlabel="time, s", ylabel="mean |ΔI|, grey levels", title="Motion activity of the fragment")
axes[0].legend(loc="upper right")
fs_grid = np.arange(0.5, 15.01, 0.5)
n_theory = np.minimum(np.floor((info["frames"] - 1) * fs_grid / info["fps"] + 1e-9) + 1, info["frames"])
axes[1].plot(fs_grid, n_theory, color=COLORS[2], lw=2)
axes[1].axvline(info["fps"], color="grey", ls="--", lw=1)
axes[1].text(info["fps"], n_theory.max() * 0.5, "  f_s = f_v", color="grey")
axes[1].scatter([SAMPLING_FPS], [len(frames_table)], color=COLORS[1], zorder=3, label="this run")
axes[1].set(xlabel="sampling rate f_s, frames/s", ylabel="saved frames N_s", title="Number of frames against f_s")
axes[1].legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "1_2_sampling.png")

saved = [cv2.cvtColor(cv2.imread(os.path.join(DIRS["frames"], f)), cv2.COLOR_BGR2RGB) for f in frames_table.frame[:18]]
sheet = aitbc.contact_sheet(saved, cols=6, width=1500, labels=[f"t = {t:.1f} s" for t in frames_table.time_s[:18]])
fig, ax = plt.subplots(figsize=(15, 15 * sheet.shape[0] / sheet.shape[1]))
ax.imshow(sheet); ax.axis("off"); ax.set_title("Saved frames (first 18)")
save_figure(fig, "1_2_contact_sheet.png")

**Task 1.**
1. Change `SAMPLING_FPS` to 0.5, 2, 5 and 10. Compare the number of files and the interval between samples with the formula for $N_s$.
2. Find on the motion-activity plot the moments of the largest change of the scene. At which $f_s$ do they still fall between the saved frames?
3. Repeat the decomposition for the thermal channel of the same sequence (1.1, `CHANNEL = thermal`). Is the motion activity of the thermal video higher or lower, and why?
4. To process your own video, upload it to Google Drive and pass its path to `extract_frames`.

---
## Topic 2. The digital image and its preprocessing

A colour frame is a three-dimensional array $H \times W \times 3$ of 8-bit numbers (0–255): rows, columns and the channels R, G, B. A thermal frame is a two-dimensional array $H \times W$ of grey levels. Every later step (filters, features, neural networks) works with these numbers, therefore their distribution, noise and scale define what a model can see.

### 2.1. Colour channels, histograms and object sizes

**Purpose.** Load a real VisDrone frame with annotated pedestrians and vehicles, look at its channels and histograms, and measure how many pixels an object occupies.

**Parameters.**
- `IMAGE_NUMBER` — one of ten selected VisDrone frames (different heights, view angles and scenes).
- `WORK_WIDTH` — width of the working copy, px. The frame and its boxes are scaled together; a smaller width speeds up the following blocks but makes small objects smaller.

**What to look at.**
1. The histograms: a narrow histogram means low contrast; peaks at 0 or 255 mean clipping (lost details in shadows or highlights).
2. The table of channel statistics: the entropy $H = -\sum_i p_i \log_2 p_i$ (bits) measures how many grey levels the channel actually uses (maximum 8 bits).
3. The distribution of the object size $\sqrt{w h}$: in drone images most objects are only 10–40 px across. This is the main difficulty of detection from a UAV.

Note: OpenCV reads files in the order **BGR**; after `cv2.imread` the channels must be converted with `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)`. The course library returns RGB images.

In [ ]:
#@title 2.1 Colour channels, histograms and object sizes { display-mode: "form" }
IMAGE_NUMBER = 1 #@param {type:"slider", min:1, max:10, step:1}
WORK_WIDTH = 960 #@param {type:"slider", min:480, max:1600, step:80}

IMAGE_NAME = vis.CURATED[IMAGE_NUMBER - 1]
full_rgb, full_objects = vis.load(IMAGE_NAME)
k_scale = WORK_WIDTH / full_rgb.shape[1]
image_rgb = cv2.resize(full_rgb, (WORK_WIDTH, int(round(full_rgb.shape[0] * k_scale))),
                       interpolation=cv2.INTER_AREA if k_scale < 1 else cv2.INTER_LINEAR)
image_objects = [dict(o, box=[v * k_scale for v in o["box"]]) for o in full_objects]
image_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
IMAGE_SOURCE = f"VisDrone2019-DET val, {IMAGE_NAME}"
print(f"{IMAGE_SOURCE}: original {full_rgb.shape[1]} × {full_rgb.shape[0]} px, working copy "
      f"{image_rgb.shape[1]} × {image_rgb.shape[0]} px, array {image_rgb.shape}, type {image_rgb.dtype}, "
      f"{len(image_objects)} annotated objects")


def entropy_bits(channel):
    p = np.bincount(channel.ravel(), minlength=256) / channel.size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


stats = []
for i, name in enumerate("RGB"):
    c = image_rgb[:, :, i]
    stats.append(dict(Channel=name, Mean=c.mean(), Std=c.std(), P1=np.percentile(c, 1), P99=np.percentile(c, 99),
                      **{"Entropy, bits": entropy_bits(c)}, **{"Clipped, %": 100 * np.mean((c == 0) | (c == 255))}))
stats.append(dict(Channel="Grey", Mean=image_gray.mean(), Std=image_gray.std(), P1=np.percentile(image_gray, 1),
                  P99=np.percentile(image_gray, 99), **{"Entropy, bits": entropy_bits(image_gray)},
                  **{"Clipped, %": 100 * np.mean((image_gray == 0) | (image_gray == 255))}))
display(pd.DataFrame(stats).set_index("Channel").round(2))

fig = plt.figure(figsize=(15, 6.2))
grid = fig.add_gridspec(3, 2, width_ratios=[2.6, 1], hspace=0.25, wspace=0.03)
ax = fig.add_subplot(grid[:, 0])
ax.imshow(aitbc.draw_boxes(image_rgb, image_objects, colors=vis.COLORS, labels=False, thickness=1))
present = sorted({o["cls"] for o in image_objects})
aitbc.legend(ax, present, [vis.COLORS[c] for c in present], ncol=2)
ax.set_title(f"{IMAGE_SOURCE}: {len(image_objects)} annotated objects"); ax.axis("off")
for i, name in enumerate("RGB"):
    ax = fig.add_subplot(grid[i, 1])
    ax.imshow(image_rgb[:, :, i], cmap="gray", vmin=0, vmax=255); ax.axis("off")
    ax.set_title(f"channel {name} as grey levels", fontsize=9)
save_figure(fig, "2_1_frame_channels.png")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
ax = axes[0]
for i, (name, color) in enumerate(zip("RGB", ["#c0392b", "#27ae60", "#2e6db4"])):
    h = np.bincount(image_rgb[:, :, i].ravel(), minlength=256) / image_gray.size
    ax.plot(np.arange(256), h, color=color, lw=1.3, label=name)
ax2 = ax.twinx()
ax2.plot(np.arange(256), np.cumsum(np.bincount(image_gray.ravel(), minlength=256)) / image_gray.size,
         color="k", lw=1, ls="--", label="cumulative (grey)")
ax2.set_ylabel("cumulative share"); ax2.spines["right"].set_visible(True)
ax.set(xlabel="brightness level", ylabel="relative frequency", title="Histograms of the channels")
ax.legend(loc="upper left"); ax2.legend(loc="center right")
ax = axes[1]
sizes = np.array([math.sqrt(max(o["box"][2] - o["box"][0], 1) * max(o["box"][3] - o["box"][1], 1)) for o in image_objects])
groups = {"persons": ("pedestrian", "people"), "two-wheelers": ("bicycle", "motor", "tricycle", "awning-tricycle"),
          "cars and vans": ("car", "van"), "trucks and buses": ("truck", "bus")}
bins = np.linspace(0, max(80, sizes.max() + 5), 25)
data = [[z for z, o in zip(sizes, image_objects) if o["cls"] in members] for members in groups.values()]
ax.hist([d for d in data if d], bins=bins, stacked=True, color=[c for d, c in zip(data, COLORS) if d],
        label=[f"{g} ({len(d)})" for g, d in zip(groups, data) if d])
ax.axvline(np.median(sizes), color="k", ls="--", lw=1)
ax.text(np.median(sizes), ax.get_ylim()[1] * 0.92, f"  median {np.median(sizes):.0f} px")
ax.set(xlabel="object size \u221a(w\u00b7h), px", ylabel="number of objects", title="How many pixels an object occupies (stacked)")
ax.legend()
fig.tight_layout(w_pad=4)
save_figure(fig, "2_1_histograms_sizes.png")

### 2.2. Degradations of the image and noise filters

**Purpose.** Reproduce typical degradations of a UAV video (sensor noise, motion blur, compression in the radio link) and compare four noise filters by objective metrics.

The metrics compare the restored image $\hat I$ with the known undistorted frame $I$:
- **PSNR** (peak signal-to-noise ratio), dB: $\mathrm{PSNR} = 10 \log_{10} \dfrac{255^2}{\mathrm{MSE}}$, where MSE is the mean squared difference of the pixels. +6 dB means half the error amplitude.
- **SSIM** (structural similarity, 0…1): compares local brightness, contrast and structure; it is closer to visual quality than PSNR.
- **Sharpness**: variance of the Laplacian; filters that remove noise also lower it.

| Filter | Principle | Typical property |
|---|---|---|
| Gaussian | weighted mean of a $k \times k$ window | fast, blurs edges |
| Median | median of a $k \times k$ window | removes impulse noise, keeps edges |
| Bilateral | weights by distance **and** by brightness difference | keeps edges, slow on large windows |
| Non-local means | averages similar patches of the whole neighbourhood | best quality, the slowest |

**Parameters.** `NOISE_SIGMA` — standard deviation of the Gaussian sensor noise, grey levels; `MOTION_BLUR_PX` — length of the motion blur, px; `JPEG_QUALITY` — JPEG quality of the video link (100 = no compression); `KERNEL` — window size $k$ of the filters; `NLM_STRENGTH` — strength $h$ of non-local means.

**What to look at.** The table and the enlarged fragment with small objects; the curves PSNR($\sigma$) and SSIM($\sigma$): which filter is best at weak noise and which one at strong noise.

In [ ]:
#@title 2.2 Degradations of the image and noise filters { display-mode: "form" }
NOISE_SIGMA = 20 #@param {type:"slider", min:0, max:60, step:2}
MOTION_BLUR_PX = 0 #@param {type:"slider", min:0, max:20, step:1}
JPEG_QUALITY = 100 #@param {type:"slider", min:10, max:100, step:5}
KERNEL = 5 #@param [3, 5, 7, 9] {type:"raw"}
NLM_STRENGTH = 10 #@param {type:"slider", min:3, max:30, step:1}

degraded_rgb = aitbc.degrade(image_rgb, motion_px=MOTION_BLUR_PX, noise=NOISE_SIGMA, jpeg_quality=JPEG_QUALITY, seed=SEED)


def apply_filters(img, k=KERNEL, h=NLM_STRENGTH, sigma=NOISE_SIGMA):
    return {
        "Gaussian": cv2.GaussianBlur(img, (k, k), 0),
        "Median": cv2.medianBlur(img, k),
        "Bilateral": cv2.bilateralFilter(img, k, 2.0 * sigma + 10.0, k),
        "Non-local means": cv2.fastNlMeansDenoisingColored(img, None, h, h, 7, 21),
    }


t0 = time.time()
filtered = apply_filters(degraded_rgb)
rows = [dict(Method="Degraded frame", **{"PSNR, dB": aitbc.psnr(image_rgb, degraded_rgb), "SSIM": aitbc.ssim(image_rgb, degraded_rgb),
             "Sharpness": aitbc.sharpness(degraded_rgb)})]
for name, img in filtered.items():
    rows.append(dict(Method=name, **{"PSNR, dB": aitbc.psnr(image_rgb, img), "SSIM": aitbc.ssim(image_rgb, img),
                                     "Sharpness": aitbc.sharpness(img)}))
restoration_table = pd.DataFrame(rows).set_index("Method")
restoration_table.loc["Undistorted frame (reference)"] = [np.inf, 1.0, aitbc.sharpness(image_rgb)]
display(restoration_table.round(3))
restoration_table.to_csv(os.path.join(OUT, "restoration_metrics.csv"))

# enlarged fragment around the densest group of objects
centres = np.array([[(o["box"][0] + o["box"][2]) / 2, (o["box"][1] + o["box"][3]) / 2] for o in image_objects])
H_img, W_img = image_gray.shape
cw, ch = W_img // 4, H_img // 4
if len(centres):
    best = max(centres, key=lambda c: np.sum((np.abs(centres[:, 0] - c[0]) < cw / 2) & (np.abs(centres[:, 1] - c[1]) < ch / 2)))
else:
    best = (W_img / 2, H_img / 2)
x0 = int(np.clip(best[0] - cw / 2, 0, W_img - cw)); y0 = int(np.clip(best[1] - ch / 2, 0, H_img - ch))
crop = lambda im: cv2.resize(im[y0:y0 + ch, x0:x0 + cw], (cw * 3, ch * 3), interpolation=cv2.INTER_NEAREST)
panels = [image_rgb, degraded_rgb] + list(filtered.values())
names = ["Reference", f"Degraded: σ = {NOISE_SIGMA}, blur {MOTION_BLUR_PX} px, JPEG {JPEG_QUALITY}"] + \
        [f"{n}: {restoration_table.loc[n, 'PSNR, dB']:.1f} dB, SSIM {restoration_table.loc[n, 'SSIM']:.2f}" for n in filtered]
show_images([crop(p) for p in panels], names, "2_2_filters_fragment.png", cols=3, width=4.6)
print(f"Filters computed in {time.time() - t0:.1f} s; fragment {cw} × {ch} px at x = {x0}, y = {y0}, enlarged 3 times")

# sweep of the noise level (noise only, the other degradations off)
sigmas = [0, 5, 10, 20, 30, 45, 60]
sweep = []
for s in sigmas:
    noisy = aitbc.add_noise(image_rgb, s, seed=SEED) if s > 0 else image_rgb.copy()
    sweep.append(dict(sigma=s, method="No filter", psnr=aitbc.psnr(image_rgb, noisy), ssim=aitbc.ssim(image_rgb, noisy)))
    for name, img in apply_filters(noisy, sigma=max(s, 1), h=max(3, 0.6 * s)).items():
        sweep.append(dict(sigma=s, method=name, psnr=aitbc.psnr(image_rgb, img), ssim=aitbc.ssim(image_rgb, img)))
sweep = pd.DataFrame(sweep)
sweep.to_csv(os.path.join(OUT, "restoration_sweep.csv"), index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (name, g), color, marker in zip(sweep.groupby("method", sort=False), ["k"] + COLORS, "osD^v"):
    axes[0].plot(g.sigma, g.psnr.clip(upper=50), marker=marker, color=color, label=name)
    axes[1].plot(g.sigma, g.ssim, marker=marker, color=color, label=name)
axes[0].set(xlabel="noise σ, grey levels", ylabel="PSNR, dB (clipped at 50)", title=f"PSNR against noise, window k = {KERNEL}")
axes[1].set(xlabel="noise σ, grey levels", ylabel="SSIM", title="SSIM against noise (h of NL-means = 0.6 σ)")
for ax in axes:
    ax.axvline(NOISE_SIGMA, color="grey", ls=":", lw=1)
    ax.grid(alpha=0.3)
axes[1].legend()
fig.tight_layout()
save_figure(fig, "2_2_noise_sweep.png")

### 2.3. Low light, contrast enhancement and letterbox

**Purpose.** Simulate a frame recorded at dusk (less light reaches the sensor, the photon noise grows relative to the signal), compare three methods of contrast enhancement, and prepare the frame for a neural network with a fixed input size (letterbox).

| Method | Principle |
|---|---|
| Gamma correction | $I_{out} = 255\,(I_{in}/255)^{\gamma}$, $\gamma < 1$ brightens dark tones |
| Global histogram equalisation | one brightness mapping for the whole frame, the histogram becomes flat |
| **CLAHE** | equalisation in tiles of a $n \times n$ grid with a clip limit of the histogram; limits the amplification of noise |

The enhancement is applied to the lightness channel $L$ of the colour space LAB, so that the colours are not distorted.

**Letterbox.** A detector such as YOLO takes a square input, e.g. 640 × 640. The frame is scaled with the factor $r = \min(S/W,\ S/H)$ without changing the proportions and the free area is filled with grey. The boxes of the objects must be transformed in the same way: $x' = r\,x + \Delta x$, $y' = r\,y + \Delta y$.

**Parameters.** `LIGHT_LEVEL` — share of the light that reaches the sensor (1 = normal day); `GAMMA`; `CLAHE_CLIP` — clip limit; `CLAHE_TILES` — size $n$ of the tile grid; `TARGET_SIZE` — input size $S$ of the network.

**What to look at.**
1. The table: SSIM with the normal frame and the noise in a flat area. A stronger enhancement makes the frame brighter, and the noise brighter too.
2. The plot "objects smaller than 8 px against the input size": at small $S$ most VisDrone objects become too small for any detector.

In [ ]:
#@title 2.3 Low light, contrast enhancement and letterbox { display-mode: "form" }
LIGHT_LEVEL = 0.08 #@param {type:"slider", min:0.02, max:0.5, step:0.01}
GAMMA = 0.5 #@param {type:"slider", min:0.2, max:1.0, step:0.05}
CLAHE_CLIP = 2.5 #@param {type:"slider", min:1.0, max:8.0, step:0.5}
CLAHE_TILES = 8 #@param [2, 4, 8, 16] {type:"raw"}
TARGET_SIZE = 640 #@param [320, 416, 640, 960, 1280] {type:"raw"}

dark_rgb = aitbc.low_light(image_rgb, level=LIGHT_LEVEL, gain_max=1.0, seed=SEED)   # no automatic gain


def on_lightness(rgb, fn):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = fn(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def clahe(rgb, clip=CLAHE_CLIP, tiles=CLAHE_TILES):
    return on_lightness(rgb, cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply)


gamma_lut = (255.0 * (np.arange(256) / 255.0) ** GAMMA).astype(np.uint8)
enhanced = {
    f"Gamma γ = {GAMMA}": on_lightness(dark_rgb, lambda L: cv2.LUT(L, gamma_lut)),
    "Global equalisation": on_lightness(dark_rgb, cv2.equalizeHist),
    f"CLAHE clip {CLAHE_CLIP}, grid {CLAHE_TILES}×{CLAHE_TILES}": clahe(dark_rgb),
}
# flat area for the noise estimate: the 10 % of 16x16 blocks with the smallest gradient in the normal frame
grad = cv2.blur(np.abs(cv2.Laplacian(image_gray, cv2.CV_32F)), (16, 16))
flat = grad <= np.percentile(grad, 10)


def noise_in_flat(rgb):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    return float((g - cv2.GaussianBlur(g, (0, 0), 2.0))[flat].std())


rows = []
for name, img in [("Normal frame", image_rgb), (f"Dark frame, light {LIGHT_LEVEL:.2f}", dark_rgb)] + list(enhanced.items()):
    L = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:, :, 0]
    rows.append({"Method": name, "Mean L": L.mean(), "RMS contrast": L.std(), "Entropy, bits": entropy_bits(L),
                 "SSIM with normal": aitbc.ssim(image_rgb, img), "Noise in flat area": noise_in_flat(img)})
contrast_table = pd.DataFrame(rows).set_index("Method")
display(contrast_table.round(3))
contrast_table.to_csv(os.path.join(OUT, "contrast_metrics.csv"))
show_images([dark_rgb] + list(enhanced.values()), [f"Dark frame, light {LIGHT_LEVEL:.2f}"] + list(enhanced),
            "2_3_contrast.png", cols=4, width=3.8)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for (name, img), color in zip([("Normal", image_rgb), ("Dark", dark_rgb)] + list(enhanced.items()), ["k", "grey"] + COLORS):
    L = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:, :, 0]
    axes[0].plot(np.bincount(L.ravel(), minlength=256) / L.size, color=color, lw=1.2, label=name)
axes[0].set(xlabel="lightness L", ylabel="relative frequency", title="Histograms of L", ylim=(0, None))
axes[0].legend(fontsize=8)
for tiles, color in zip([2, 4, 8, 16], COLORS):
    clips = np.arange(1.0, 8.01, 1.0)
    res = [clahe(dark_rgb, c, tiles) for c in clips]
    axes[1].plot(clips, [aitbc.ssim(image_rgb, r) for r in res], marker="o", color=color, label=f"grid {tiles}×{tiles}")
axes[1].set(xlabel="CLAHE clip limit", ylabel="SSIM with the normal frame", title="Tuning of CLAHE on the dark frame")
axes[1].axvline(CLAHE_CLIP, color="grey", ls=":"); axes[1].grid(alpha=0.3); axes[1].legend()
fig.tight_layout()
save_figure(fig, "2_3_clahe_tuning.png")


def letterbox(rgb, size=640, color=(114, 114, 114)):
    # scale without changing the proportions and pad to size x size; returns the image and the parameters
    h, w = rgb.shape[:2]
    r = min(size / w, size / h)
    nw, nh = max(1, round(w * r)), max(1, round(h * r))
    resized = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA if r < 1 else cv2.INTER_LINEAR)
    left, top = (size - nw) // 2, (size - nh) // 2
    out = np.full((size, size, 3), color, np.uint8)
    out[top:top + nh, left:left + nw] = resized
    return out, dict(ratio=r, left=left, top=top)


def letterbox_boxes(objects, p):
    return [dict(o, box=[o["box"][0] * p["ratio"] + p["left"], o["box"][1] * p["ratio"] + p["top"],
                         o["box"][2] * p["ratio"] + p["left"], o["box"][3] * p["ratio"] + p["top"]]) for o in objects]


lb_rgb, lb_info = letterbox(full_rgb, TARGET_SIZE)
lb_objects = letterbox_boxes(full_objects, lb_info)
min_side = lambda objs: np.array([min(o["box"][2] - o["box"][0], o["box"][3] - o["box"][1]) for o in objs])
sizes_S = np.array([256, 320, 416, 512, 640, 800, 960, 1280, 1600, 2048])
share_small = [100 * np.mean(min_side(letterbox_boxes(full_objects, dict(ratio=min(S / full_rgb.shape[1], S / full_rgb.shape[0]),
                                                                          left=0, top=0))) < 8) for S in sizes_S]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
axes[0].imshow(aitbc.draw_boxes(lb_rgb, lb_objects, colors=vis.COLORS, labels=False))
axes[0].set_title(f"Letterbox {TARGET_SIZE} × {TARGET_SIZE}: r = {lb_info['ratio']:.3f}, "
                  f"Δx = {lb_info['left']}, Δy = {lb_info['top']} px"); axes[0].axis("off")
axes[1].plot(sizes_S, share_small, marker="o", color=COLORS[1])
axes[1].axvline(TARGET_SIZE, color="grey", ls=":")
axes[1].set(xscale="log", xlabel="input size S of the network, px", ylabel="objects with a side < 8 px, %",
            title=f"Small objects after scaling (original frame {full_rgb.shape[1]} × {full_rgb.shape[0]} px)")
axes[1].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); axes[1].xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
axes[1].set_xticks(sizes_S); axes[1].set_xticklabels(sizes_S, rotation=45); axes[1].grid(alpha=0.3)
fig.tight_layout()
save_figure(fig, "2_3_letterbox.png")

**Task 2.**
1. Set `NOISE_SIGMA` to 5 and to 50. Which filter has the highest PSNR in each case? Does the order of the filters by SSIM coincide with the order by PSNR?
2. Add a motion blur of 8 px. Why do the noise filters not restore a blurred frame?
3. Change `CLAHE_CLIP` from 1 to 8 and look at the column "Noise in flat area". Explain why a better visual impression does not guarantee a higher accuracy of the next model.
4. With the plot of small objects choose the input size of the detector for this frame. What is the price of a large input size?

---
## Topic 3. Classical methods of feature extraction

Before neural networks, computer vision was built on hand-designed features: brightness gradients, edges, corners and local descriptors. They are still used on board of UAVs because they are fast, need no training and are easy to interpret: visual odometry, image stitching, stabilisation of the video, detection of moving objects.

### 3.1. Gradients, edges and contours

**Purpose.** Compute the brightness gradient with the Sobel operator, find edges with the Canny detector and outline the contours.

The Sobel operator estimates the derivatives $G_x = \partial I/\partial x$, $G_y = \partial I/\partial y$; the gradient magnitude $|G| = \sqrt{G_x^2 + G_y^2}$ is large at edges and the orientation $\theta = \operatorname{atan2}(G_y, G_x)$ shows their direction. The Canny detector keeps the local maxima of $|G|$ and uses two thresholds: pixels above `CANNY_HIGH` are strong edges, pixels between `CANNY_LOW` and `CANNY_HIGH` are kept when they are connected to strong edges (hysteresis).

**Parameters.**
- `BLUR_SIGMA` — Gaussian smoothing before Canny (suppresses noise and fine texture);
- `AUTO_THRESHOLDS` — thresholds from the median $m$ of the frame: $T_{low} = 0.66\,m$, $T_{high} = 1.33\,m$;
- `CANNY_LOW`, `CANNY_HIGH` — manual thresholds; `MIN_CONTOUR_LENGTH` — shortest contour to draw, px.

**What to look at.** The colour map of the orientation (hue = direction of the edge); the curve of edge density against the threshold; the edge density inside the object boxes and in the background: vehicles and people are richer in edges than the road surface, therefore edges are a first cue for detection.

In [ ]:
#@title 3.1 Gradients, edges and contours { display-mode: "form" }
BLUR_SIGMA = 1.0 #@param {type:"slider", min:0, max:4, step:0.25}
AUTO_THRESHOLDS = False #@param {type:"boolean"}
CANNY_LOW = 60 #@param {type:"slider", min:0, max:250, step:5}
CANNY_HIGH = 150 #@param {type:"slider", min:10, max:255, step:5}
MIN_CONTOUR_LENGTH = 60 #@param {type:"slider", min:10, max:300, step:10}

smooth = cv2.GaussianBlur(image_gray, (0, 0), BLUR_SIGMA) if BLUR_SIGMA > 0 else image_gray
if AUTO_THRESHOLDS:
    m = float(np.median(smooth))
    low, high = int(max(0, 0.66 * m)), int(min(255, 1.33 * m))
else:
    low, high = CANNY_LOW, CANNY_HIGH
if low >= high:
    raise ValueError("The low threshold of Canny must be smaller than the high threshold.")
gx = cv2.Sobel(smooth, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)
magnitude, angle = cv2.cartToPolar(gx, gy, angleInDegrees=True)
edges = cv2.Canny(smooth, low, high)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
long_contours = [c for c in contours if cv2.arcLength(c, False) > MIN_CONTOUR_LENGTH]
contours_rgb = image_rgb.copy()
cv2.drawContours(contours_rgb, long_contours, -1, (255, 60, 40), 1)

# orientation map: hue = direction, value = magnitude
hsv = np.zeros(image_rgb.shape, np.uint8)
hsv[:, :, 0] = (angle % 180).astype(np.uint8)
hsv[:, :, 1] = 255
hsv[:, :, 2] = np.clip(magnitude / np.percentile(magnitude, 99) * 255, 0, 255).astype(np.uint8)
orientation_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
show_images([np.clip(magnitude / np.percentile(magnitude, 99), 0, 1), orientation_rgb, edges, contours_rgb],
            ["Sobel gradient magnitude |G|", "Orientation θ (hue) and |G| (brightness)",
             f"Canny edges, thresholds {low} / {high}", f"{len(long_contours)} contours longer than {MIN_CONTOUR_LENGTH} px"],
            "3_1_edges.png", cols=2, width=6.2)

object_mask = np.zeros(image_gray.shape, bool)
for o in image_objects:
    x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
    object_mask[max(y1, 0):y2, max(x1, 0):x2] = True
highs = np.arange(20, 256, 10)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
for ratio, color in zip([0.3, 0.5, 0.8], COLORS):
    axes[0].plot(highs, [100 * np.mean(cv2.Canny(smooth, int(ratio * h), int(h)) > 0) for h in highs],
                 color=color, marker=".", label=f"T_low = {ratio} · T_high")
axes[0].scatter([high], [100 * np.mean(edges > 0)], color="k", zorder=3, label="this run")
axes[0].set(xlabel="high threshold T_high", ylabel="edge pixels, %", title="Edge density against the Canny threshold")
axes[0].grid(alpha=0.3); axes[0].legend()
inside, outside = 100 * np.mean(edges[object_mask] > 0), 100 * np.mean(edges[~object_mask] > 0)
axes[1].bar(["inside object boxes", "background"], [inside, outside], color=[COLORS[1], COLORS[0]])
for i, v in enumerate([inside, outside]):
    axes[1].text(i, v, f"{v:.1f} %", ha="center", va="bottom")
axes[1].set(ylabel="edge pixels, %", title=f"Edges on objects and on the background ({inside / max(outside, 1e-6):.1f} ×)")
fig.tight_layout()
save_figure(fig, "3_1_edge_density.png")

### 3.2. ORB features and estimation of a homography

**Purpose.** Find corresponding points of two views of the same scene and estimate the geometric transformation between them.

**ORB** (Oriented FAST and Rotated BRIEF) finds corners (FAST) and describes the neighbourhood of every corner by a 256-bit binary string (BRIEF), rotated by the dominant direction of the corner. Descriptors are compared by the Hamming distance (number of different bits). A match is accepted by the **ratio test**: the best distance must be clearly smaller than the second best, $d_1 < \rho\, d_2$.

A plane scene viewed by two cameras is related by a **homography** — a $3 \times 3$ matrix $H$:
$$\begin{pmatrix} x' \\ y' \\ 1 \end{pmatrix} \sim H \begin{pmatrix} x \\ y \\ 1 \end{pmatrix}.$$
**RANSAC** repeatedly estimates $H$ from 4 random matches and keeps the estimate with the largest number of matches (inliers) whose reprojection error is below `RANSAC_PX`.

In this block the second view is produced from the frame by a **known** homography $H_{true}$ (rotation, scale and perspective tilt). This makes it possible to measure the error exactly: the corner error is the mean distance between the frame corners transformed by $H_{est}$ and by $H_{true}$.

**Parameters.** `ORB_FEATURES` — number of ORB corners; `RATIO_TEST` — $\rho$; `RANSAC_PX` — inlier threshold; `ROTATION_DEG`, `SCALE`, `TILT` — the known transformation (tilt = perspective, as when the camera is inclined).

**What to look at.** Inlier matches (green) and rejected matches (red); the corner error; the robustness curves: ORB descriptors are rotated with the corner, so the number of inliers hardly depends on rotation, but it falls quickly when the scale changes by more than 1.5–2 times.

In [ ]:
#@title 3.2 ORB features and homography with a known transformation { display-mode: "form" }
ORB_FEATURES = 2000 #@param {type:"slider", min:250, max:6000, step:250}
RATIO_TEST = 0.75 #@param {type:"slider", min:0.5, max:0.95, step:0.05}
RANSAC_PX = 3.0 #@param {type:"slider", min:0.5, max:10, step:0.5}
ROTATION_DEG = 20 #@param {type:"slider", min:-90, max:90, step:5}
SCALE = 0.9 #@param {type:"slider", min:0.4, max:2.0, step:0.05}
TILT = 2.0 #@param {type:"slider", min:0, max:6, step:0.5}

H_img, W_img = image_gray.shape
corners = np.float32([[0, 0], [W_img, 0], [W_img, H_img], [0, H_img]]).reshape(-1, 1, 2)


def known_homography(rotation_deg, scale, tilt):
    # rotation and scale about the frame centre, then a perspective tilt (units 1e-4 per px)
    c, s = math.cos(math.radians(rotation_deg)), math.sin(math.radians(rotation_deg))
    to_c = np.array([[1, 0, -W_img / 2], [0, 1, -H_img / 2], [0, 0, 1]])
    rs = np.array([[scale * c, -scale * s, 0], [scale * s, scale * c, 0], [0, 0, 1]])
    persp = np.array([[1, 0, 0], [0, 1, 0], [0, tilt * 1e-4, 1]])
    back = np.array([[1, 0, W_img / 2], [0, 1, H_img / 2], [0, 0, 1]])
    return back @ persp @ rs @ to_c


def match_orb(gray_a, gray_b, n_features=ORB_FEATURES, ratio=RATIO_TEST, ransac_px=RANSAC_PX):
    orb = cv2.ORB_create(nfeatures=n_features)
    kp1, des1 = orb.detectAndCompute(gray_a, None)
    kp2, des2 = orb.detectAndCompute(gray_b, None)
    out = dict(kp1=kp1, kp2=kp2, good=[], H=None, inliers=np.zeros(0, bool))
    if des1 is None or des2 is None:
        return out
    pairs = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(des1, des2, k=2)
    good = [p[0] for p in pairs if len(p) == 2 and p[0].distance < ratio * p[1].distance]
    out["good"] = good
    if len(good) >= 8:
        src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        H, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_px)
        if H is not None:
            out.update(H=H, inliers=mask.ravel().astype(bool), src=src, dst=dst)
    return out


def corner_error(H_est, H_true):
    if H_est is None:
        return np.nan
    a = cv2.perspectiveTransform(corners, H_est)
    b = cv2.perspectiveTransform(corners, H_true.astype(np.float32))
    return float(np.linalg.norm(a - b, axis=2).mean())


H_true = known_homography(ROTATION_DEG, SCALE, TILT)
warped_rgb = cv2.warpPerspective(image_rgb, H_true, (W_img, H_img), borderMode=cv2.BORDER_CONSTANT)
res = match_orb(image_gray, cv2.cvtColor(warped_rgb, cv2.COLOR_RGB2GRAY))
if res["H"] is None:
    print(f"The homography was not estimated: {len(res['good'])} matches after the ratio test.")
else:
    inl = res["inliers"]
    reproj = np.linalg.norm(cv2.perspectiveTransform(res["src"][inl], res["H"]) - res["dst"][inl], axis=2).ravel()
    orb_table = pd.DataFrame([("ORB corners in the frames", f"{len(res['kp1'])} / {len(res['kp2'])}"),
                              ("Matches after the ratio test", len(res["good"])),
                              ("RANSAC inliers", f"{inl.sum()} ({100 * inl.mean():.0f} %)"),
                              ("Median reprojection error of inliers", f"{np.median(reproj):.2f} px"),
                              ("Corner error of H_est against H_true", f"{corner_error(res['H'], H_true):.2f} px")],
                             columns=["Quantity", "Value"]).set_index("Quantity")
    display(orb_table)
    sel = np.random.default_rng(SEED).choice(len(res["good"]), size=min(80, len(res["good"])), replace=False)
    vis_matches = cv2.drawMatches(image_rgb, res["kp1"], warped_rgb, res["kp2"], [res["good"][i] for i in sel if inl[i]], None,
                                  matchColor=(40, 220, 60), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    vis_matches = cv2.drawMatches(image_rgb, res["kp1"], warped_rgb, res["kp2"], [res["good"][i] for i in sel if not inl[i]],
                                  vis_matches, matchColor=(255, 40, 40),
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS | cv2.DrawMatchesFlags_DRAW_OVER_OUTIMG)
    show_images([vis_matches], [f"ORB matches (80 shown): green = inliers, red = rejected by RANSAC; "
                                f"rotation {ROTATION_DEG}°, scale {SCALE}, tilt {TILT}"], "3_2_orb_matches.png", width=14)

# robustness of ORB to rotation and scale
rows = []
for r in range(-90, 91, 15):
    H = known_homography(r, 1.0, 0.0)
    out = match_orb(image_gray, cv2.warpPerspective(image_gray, H, (W_img, H_img)))
    rows.append(dict(kind="rotation", value=r, inliers=int(out["inliers"].sum()), error=corner_error(out["H"], H)))
for s in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]:
    H = known_homography(0, s, 0.0)
    out = match_orb(image_gray, cv2.warpPerspective(image_gray, H, (W_img, H_img)))
    rows.append(dict(kind="scale", value=s, inliers=int(out["inliers"].sum()), error=corner_error(out["H"], H)))
robust = pd.DataFrame(rows)
robust.to_csv(os.path.join(OUT, "orb_robustness.csv"), index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, kind, xlabel in [(axes[0], "rotation", "rotation, degrees"), (axes[1], "scale", "scale factor")]:
    g = robust[robust.kind == kind]
    ax.plot(g.value, g.inliers, marker="o", color=COLORS[0])
    ax.set(xlabel=xlabel, ylabel="RANSAC inliers", title=f"ORB against {kind}")
    ax.grid(alpha=0.3)
    ax2 = ax.twinx(); ax2.spines["right"].set_visible(True)
    ax2.semilogy(g.value, g.error.clip(lower=0.01).fillna(1e3), marker="s", ls="--", color=COLORS[1])
    ax2.set_ylabel("corner error, px (log)", color=COLORS[1])
fig.tight_layout()
save_figure(fig, "3_2_orb_robustness.png")

### 3.3. Compensation of the camera motion in a real video

**Purpose.** Apply the same method to two real frames of the UAV video from block 1.1. Even a hovering UAV drifts and sways; after the estimated homography is applied to the earlier frame, the static background cancels in the difference image and the **moving vehicles stand out**. This is the basis of the detection of moving objects from a moving camera.

**Parameters.** `FIRST_FRAME` — number of the first frame inside the fragment; `FRAME_GAP` — distance to the second frame (frames; at 10 frames/s a gap of 10 is 1 s).

**What to look at.** The difference images without and with compensation; the table with the mean residual; the annotated vehicles (yellow boxes) on the compensated difference image.

In [ ]:
#@title 3.3 Compensation of the camera motion in the UAV video { display-mode: "form" }
FIRST_FRAME = 1 #@param {type:"slider", min:1, max:150, step:1}
FRAME_GAP = 10 #@param {type:"slider", min:1, max:50, step:1}

i1 = int(min(FIRST_FRAME - 1, count - 2))
i2 = int(min(i1 + FRAME_GAP, count - 1))
fa, fb = FRAMES[i1], FRAMES[i2]
ga, gb = cv2.cvtColor(fa, cv2.COLOR_RGB2GRAY), cv2.cvtColor(fb, cv2.COLOR_RGB2GRAY)
res = match_orb(ga, gb, n_features=3000, ransac_px=2.0)
if res["H"] is None:
    raise RuntimeError("The homography between the frames was not estimated; choose other frames.")
h, w = ga.shape
aligned = cv2.warpPerspective(ga, res["H"], (w, h))
valid = cv2.warpPerspective(np.ones_like(ga), res["H"], (w, h)) > 0
diff_raw = cv2.absdiff(ga, gb)
diff_comp = cv2.absdiff(aligned, gb) * valid
shift = cv2.perspectiveTransform(np.float32([[[w / 2, h / 2]]]), res["H"])[0, 0] - [w / 2, h / 2]
objs_b = m3ot.objects(TRACKS, start + i2 + 1)
box_mask = np.zeros(gb.shape, bool)
for o in objs_b:
    x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
    box_mask[max(y1, 0):y2, max(x1, 0):x2] = True
display(pd.DataFrame([
    ("Frames", f"{start + i1 + 1} and {start + i2 + 1} ({(i2 - i1) / FPS:.1f} s apart)"),
    ("Inliers of the homography", f"{res['inliers'].sum()} of {len(res['good'])}"),
    ("Image shift at the frame centre", f"({shift[0]:.1f}, {shift[1]:.1f}) px"),
    ("Mean residual on the background, without / with compensation",
     f"{diff_raw[~box_mask].mean():.2f} / {diff_comp[valid & ~box_mask].mean():.2f} grey levels"),
    ("Mean residual inside vehicle boxes, with compensation", f"{diff_comp[box_mask & valid].mean():.2f} grey levels"),
], columns=["Quantity", "Value"]).set_index("Quantity"))
vmax = np.percentile(diff_raw, 99.5)
show_images([fb, np.clip(diff_raw / vmax, 0, 1),
             aitbc.draw_boxes(aitbc.colorize(np.clip(diff_comp / vmax * 255, 0, 255).astype(np.uint8) // 1,
                                             (plt.cm.inferno(np.linspace(0, 1, 256))[:, :3] * 255).astype(np.uint8)),
                              objs_b, color=(255, 230, 0), labels=False)],
            [f"Frame {start + i2 + 1}", "|difference| without compensation", "|difference| after compensation + annotated vehicles"],
            "3_3_motion_compensation.png", cols=3, width=4.8)

**Task 3.**
1. In block 3.2 set the rotation to 90° and then the scale to 0.5. Which change does ORB tolerate better and why?
2. Change `RATIO_TEST` from 0.6 to 0.9. How do the numbers of matches and inliers change, and what happens with the corner error?
3. In block 3.3 increase `FRAME_GAP`. Why does the residual on the background grow with the gap even after compensation? (Hint: the homography describes a plane; buildings, trees and lamp posts are not in the plane of the road.)

---
## Topic 4. Machine learning and neural networks

A classifier learns a mapping from an image (or its features) to a class from labelled examples. The quality of the learned mapping is measured on data that were **not used** for training or for the choice of the hyperparameters. In video and aerial data, neighbouring frames and objects of the same image are very similar; if they fall into both the training and the test set, the test score is too optimistic (*data leakage*). Therefore the split is made **by images**, not by individual patches.

### 4.1. Data set of thermal patches from real UAV images

**Purpose.** Build a classification data set from the annotated HIT-UAV thermal images (DJI Matrice M210 V2, Zenmuse XT2 camera, 60–130 m, day and night): square patches around persons, cars and bicycles plus random background patches without objects.

**Parameters.**
- `N_IMAGES` — number of thermal images (about 0.1 MB each). 75 % are taken from the training part of HIT-UAV and 25 % from its validation and test parts: the split by images is defined by the data set itself.
- `PATCH_SIZE` — size of a patch after scaling, px. The patch side before scaling is 1.25 × the longer side of the box.
- `BACKGROUND_PER_IMAGE` — number of background patches per image.
- `TIME_OF_DAY` — day images, night images or both.

**What to look at.** The number of patches per class (classes are not balanced, therefore macro-averaged metrics are used later); the gallery; the **mean patch** of every class — the average thermal signature that the classifier has to recognise.

In [ ]:
#@title 4.1 Data set of thermal patches (HIT-UAV) { display-mode: "form" }
N_IMAGES = 240 #@param {type:"slider", min:60, max:800, step:20}
PATCH_SIZE = 32 #@param [24, 32, 48] {type:"raw"}
BACKGROUND_PER_IMAGE = 4 #@param {type:"slider", min:0, max:12, step:1}
TIME_OF_DAY = "day and night" #@param ["day and night", "day only", "night only"]

CLASSES = ["Person", "Car", "Bicycle", "Background"]
pool = hit.images
if TIME_OF_DAY == "day only":
    pool = pool[~pool.night]
elif TIME_OF_DAY == "night only":
    pool = pool[pool.night]
pool = pool[(pool.Person + pool.Car + pool.Bicycle) > 0]
pick = np.random.default_rng(SEED)
n_train_img = int(round(0.75 * N_IMAGES))
train_pool, test_pool = pool[pool.split == "train"], pool[pool.split != "train"]
train_files = list(pick.choice(train_pool.file.values, size=min(n_train_img, len(train_pool)), replace=False))
test_files = list(pick.choice(test_pool.file.values, size=min(N_IMAGES - n_train_img, len(test_pool)), replace=False))
t0 = time.time()
hit.prefetch(train_files + test_files)


def square_crop(img, cx, cy, side, size):
    pad = int(side) + 2
    big = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    x1, y1 = int(round(cx - side / 2)) + pad, int(round(cy - side / 2)) + pad
    s = int(round(side))
    return cv2.resize(big[y1:y1 + s, x1:x1 + s], (size, size), interpolation=cv2.INTER_AREA)


patches, labels, groups, subsets, nights = [], [], [], [], []
for subset, files in [("train", train_files), ("test", test_files)]:
    for f in files:
        img, objs = hit.load(f)
        info_f = hit.images.loc[hit.images.file == f].iloc[0]
        sides = []
        for o in objs:
            if o["cls"] not in CLASSES[:3]:
                continue
            x1, y1, x2, y2 = o["box"]
            side = max(8.0, 1.25 * max(x2 - x1, y2 - y1))
            sides.append(side)
            patches.append(square_crop(img, (x1 + x2) / 2, (y1 + y2) / 2, side, PATCH_SIZE))
            labels.append(o["cls"]); groups.append(f); subsets.append(subset); nights.append(bool(info_f.night))
        # background: random squares that do not touch any annotated object
        boxes = np.array([o["box"] for o in objs], float).reshape(-1, 4)
        side_bg = float(np.median(sides)) if sides else 24.0
        placed, tries = 0, 0
        while placed < BACKGROUND_PER_IMAGE and tries < 200:
            tries += 1
            cx, cy = pick.uniform(side_bg / 2, img.shape[1] - side_bg / 2), pick.uniform(side_bg / 2, img.shape[0] - side_bg / 2)
            m = side_bg / 2 + 4
            if len(boxes) and np.any((boxes[:, 0] < cx + m) & (boxes[:, 2] > cx - m) & (boxes[:, 1] < cy + m) & (boxes[:, 3] > cy - m)):
                continue
            patches.append(square_crop(img, cx, cy, side_bg, PATCH_SIZE))
            labels.append("Background"); groups.append(f); subsets.append(subset); nights.append(bool(info_f.night))
            placed += 1

PATCHES = np.stack(patches)
LABELS = np.array(labels)
GROUPS = np.array(groups)
SUBSET = np.array(subsets)
NIGHT = np.array(nights)
y_all = np.array([CLASSES.index(c) for c in LABELS])
train_idx, test_idx = np.where(SUBSET == "train")[0], np.where(SUBSET == "test")[0]
counts = pd.crosstab(pd.Categorical(LABELS, CLASSES), SUBSET).reindex(columns=["train", "test"])
counts.index.name = "Class"
counts.loc["All"] = counts.sum()
print(f"{len(train_files)} training and {len(test_files)} test images ({TIME_OF_DAY}), "
      f"{len(PATCHES)} patches {PATCH_SIZE} × {PATCH_SIZE} px, {time.time() - t0:.0f} s")
display(counts)

fig = plt.figure(figsize=(15, 6.2))
grid = fig.add_gridspec(4, 13, width_ratios=[1] * 10 + [0.3, 1.3, 1.3])
for r, cls in enumerate(CLASSES):
    idx = np.where(LABELS == cls)[0]
    for c, i in enumerate(pick.choice(idx, size=min(10, len(idx)), replace=False)):
        ax = fig.add_subplot(grid[r, c]); ax.imshow(PATCHES[i], cmap="inferno", vmin=0, vmax=255); ax.axis("off")
        if c == 0:
            ax.set_title(cls, loc="left", fontsize=10)
    ax = fig.add_subplot(grid[r, 11]); ax.imshow(PATCHES[idx].mean(0), cmap="inferno"); ax.axis("off")
    if r == 0:
        ax.set_title("mean patch", fontsize=10)
    ax = fig.add_subplot(grid[r, 12])
    ax.hist(PATCHES[idx].reshape(len(idx), -1).mean(1), bins=20, color=COLORS[r])
    ax.set_yticks([]); ax.tick_params(labelsize=7)
    if r == 0:
        ax.set_title("mean brightness", fontsize=10)
fig.suptitle("Thermal patches (white-hot: warm = bright)", y=1.0)
fig.tight_layout()
save_figure(fig, "4_1_patches.png")

### 4.2. HOG + SVM classifier with a choice of hyperparameters

**Purpose.** Describe every patch by a **histogram of oriented gradients** (HOG) and train a support vector machine (SVM) with a radial basis kernel. The result is compared with an SVM trained on raw pixels.

**HOG.** The patch is divided into cells of $c \times c$ pixels; in every cell a histogram of the gradient orientations with `HOG_BINS` bins is built (weighted by the gradient magnitude); histograms of $2 \times 2$ neighbouring cells are normalised together (function `skimage.feature.hog`). The descriptor keeps the shape of the object and is insensitive to the absolute brightness — important for thermal images, where the brightness depends on the weather and the time of day.

**SVM.** The kernel $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \lVert \mathbf{x} - \mathbf{x}' \rVert^2)$; $C$ sets the penalty for training errors (large $C$ — a more complex boundary), $\gamma$ the width of the kernel (large $\gamma$ — a more local boundary).

**Procedure.** (1) The test images are put aside. (2) $C$ and $\gamma$ are chosen by grouped cross-validation on the training images: all patches of one image are in the same fold. (3) The chosen model is evaluated on the test images once.

**Metrics.** Accuracy is the share of correct answers; for every class precision $P = TP/(TP+FP)$, recall $R = TP/(TP+FN)$ and $F_1 = 2PR/(P+R)$; **macro** averaging takes the mean over classes, so that the rare class "Bicycle" weighs as much as "Person".

**Parameters.** `HOG_CELL` — cell size $c$, px; `HOG_BINS` — number of orientation bins; `CV_FOLDS` — number of folds.

**What to look at.** The map of the cross-validation score over $C$ and $\gamma$; the comparison HOG / pixels; the confusion matrix; the misclassified patches; the learning curve: how the quality grows with the number of training images.

In [ ]:
#@title 4.2 HOG + SVM classifier { display-mode: "form" }
HOG_CELL = 8 #@param [4, 8] {type:"raw"}
HOG_BINS = 9 #@param {type:"slider", min:6, max:18, step:3}
CV_FOLDS = 3 #@param {type:"slider", min:2, max:5, step:1}

from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from skimage.feature import hog as skimage_hog

if PATCH_SIZE % HOG_CELL:
    raise ValueError("PATCH_SIZE must be a multiple of HOG_CELL.")
hog_features = lambda p: skimage_hog(p, orientations=HOG_BINS, pixels_per_cell=(HOG_CELL, HOG_CELL),
                                     cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True)
t0 = time.time()
FEATURES = {"HOG": np.stack([hog_features(p) for p in PATCHES]).astype(np.float32),
            "Pixels": PATCHES.reshape(len(PATCHES), -1).astype(np.float32) / 255.0}
print(f"Feature length: HOG {FEATURES['HOG'].shape[1]} | pixels {FEATURES['Pixels'].shape[1]}; "
      f"HOG of {len(PATCHES)} patches computed in {time.time() - t0:.1f} s")


def fit_svm(X, idx, folds=CV_FOLDS):
    g0 = 1.0 / X.shape[1]                                   # value of gamma = "scale" after standardisation
    search = GridSearchCV(Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", class_weight="balanced"))]),
                          {"svm__C": [0.5, 2.0, 8.0, 32.0], "svm__gamma": [0.25 * g0, g0, 4 * g0]},
                          scoring="f1_macro", cv=GroupKFold(n_splits=folds), n_jobs=-1)
    search.fit(X[idx], y_all[idx], groups=GROUPS[idx])
    return search


def scores(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0)}


t0 = time.time()
SEARCH, PRED, rows = {}, {}, []
for name, X in FEATURES.items():
    SEARCH[name] = fit_svm(X, train_idx)
    PRED[name] = SEARCH[name].predict(X[test_idx])
    row = {"Features": name, **scores(y_all[test_idx], PRED[name])}
    for flag, label in [(False, "F1 day"), (True, "F1 night")]:
        sel = NIGHT[test_idx] == flag
        row[label] = f1_score(y_all[test_idx][sel], PRED[name][sel], average="macro", zero_division=0) if sel.any() else np.nan
    best = SEARCH[name].best_params_
    row["Chosen C"], row["Chosen γ·d"] = best["svm__C"], best["svm__gamma"] * X.shape[1]
    rows.append(row)
CLASSIFICATION = pd.DataFrame(rows).set_index("Features")
display(CLASSIFICATION.round(3))
CLASSIFICATION.to_csv(os.path.join(OUT, "classification_metrics.csv"))
print(f"Grid search and test in {time.time() - t0:.0f} s; γ·d is γ multiplied by the feature length d")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1, 1.1, 1]})
cv = pd.DataFrame(SEARCH["HOG"].cv_results_)
heat = cv.pivot_table(index="param_svm__C", columns="param_svm__gamma", values="mean_test_score")
im = axes[0].imshow(heat.values, cmap="viridis", origin="lower")
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(["γ₀/4", "γ₀", "4γ₀"])
axes[0].set_yticks(range(len(heat.index))); axes[0].set_yticklabels(heat.index)
for (r, c), v in np.ndenumerate(heat.values):
    axes[0].text(c, r, f"{v:.3f}", ha="center", va="center", color="w" if v < heat.values.mean() else "k")
axes[0].set(xlabel="kernel width γ (γ₀ = 1/d)", ylabel="C", title="HOG + SVM: cross-validation F1 (macro)")
cm = confusion_matrix(y_all[test_idx], PRED["HOG"], labels=range(4), normalize="true")
axes[1].imshow(cm, cmap="Greens", vmin=0, vmax=1)
for (r, c), v in np.ndenumerate(cm):
    axes[1].text(c, r, f"{v:.2f}", ha="center", va="center", color="w" if v > 0.6 else "k")
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(CLASSES, rotation=20)
axes[1].set_yticks(range(4)); axes[1].set_yticklabels(CLASSES)
axes[1].set(xlabel="predicted class", ylabel="true class", title="HOG + SVM: normalised confusion matrix (test)")
rec = {n: recall_score(y_all[test_idx], PRED[n], labels=range(4), average=None, zero_division=0) for n in FEATURES}
xx = np.arange(4)
for k, (n, r) in enumerate(rec.items()):
    axes[2].bar(xx + (k - 0.5) * 0.38, r, width=0.38, color=COLORS[k], label=n)
axes[2].set_xticks(xx); axes[2].set_xticklabels(CLASSES); axes[2].set_ylim(0, 1.05)
axes[2].set(ylabel="recall", title="Recall per class: HOG against raw pixels"); axes[2].legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "4_2_svm_results.png")

# what HOG sees, and which patches were misclassified
fig = plt.figure(figsize=(15, 4.6))
grid = fig.add_gridspec(2, 14)
for k, cls in enumerate(CLASSES):
    i = np.where(LABELS == cls)[0][0]
    _, hog_img = skimage_hog(PATCHES[i], orientations=HOG_BINS, pixels_per_cell=(HOG_CELL, HOG_CELL),
                             cells_per_block=(2, 2), visualize=True)
    ax = fig.add_subplot(grid[0, 2 * k]); ax.imshow(PATCHES[i], cmap="inferno"); ax.axis("off"); ax.set_title(cls, fontsize=9)
    ax = fig.add_subplot(grid[0, 2 * k + 1]); ax.imshow(hog_img, cmap="gray"); ax.axis("off"); ax.set_title("HOG", fontsize=9)
wrong = test_idx[PRED["HOG"] != y_all[test_idx]]
for k, i in enumerate(wrong[:14]):
    ax = fig.add_subplot(grid[1, k]); ax.imshow(PATCHES[i], cmap="inferno"); ax.axis("off")
    ax.set_title(f"{LABELS[i][:6]}→{CLASSES[PRED['HOG'][np.where(test_idx == i)[0][0]]][:6]}", fontsize=8)
fig.suptitle(f"Top: patches and their HOG (cell {HOG_CELL} px, {HOG_BINS} bins). Bottom: misclassified test patches "
             f"(true → predicted), {len(wrong)} of {len(test_idx)}", y=1.02)
save_figure(fig, "4_2_hog_and_errors.png")

# learning curve: test F1 against the number of training images
train_imgs = np.array(sorted(set(GROUPS[train_idx])))
fractions = [0.1, 0.2, 0.4, 0.7, 1.0]
curve = []
for frac in fractions:
    chosen = set(np.random.default_rng(SEED).choice(train_imgs, size=max(4, int(frac * len(train_imgs))), replace=False))
    sub = np.array([i for i in train_idx if GROUPS[i] in chosen])
    for name, X in FEATURES.items():
        p = SEARCH[name].best_params_
        model = Pipeline([("scaler", StandardScaler()),
                          ("svm", SVC(kernel="rbf", C=p["svm__C"], gamma=p["svm__gamma"], class_weight="balanced"))])
        model.fit(X[sub], y_all[sub])
        curve.append(dict(features=name, images=len(chosen), patches=len(sub),
                          f1=f1_score(y_all[test_idx], model.predict(X[test_idx]), average="macro")))
curve = pd.DataFrame(curve)
curve.to_csv(os.path.join(OUT, "learning_curve.csv"), index=False)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for (name, g), color in zip(curve.groupby("features", sort=False), COLORS):
    ax.plot(g.images, g.f1, marker="o", color=color, label=name)
ax.set(xlabel="number of training images", ylabel="test F1 (macro)", title="Learning curve of the SVM")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout()
save_figure(fig, "4_2_learning_curve.png")

### 4.3. MobileNet: transfer learning (optional block)

**Purpose.** Use a convolutional network pre-trained on ImageNet (MobileNetV3 Small, 2.5 million parameters) as a feature extractor for the **same** thermal patches and the **same** split as in 4.2, and compare it with HOG + SVM.

Transfer learning: the convolutional layers that have learned general visual features (edges, textures, shapes) are frozen; only the last classification layer (or also the last blocks) is trained on the new classes. The grey patches are scaled to `INPUT_SIZE` and repeated in three channels.

**Parameters.** `RUN_MOBILENET` — switch the block on (downloads 10 MB of weights; a GPU is advisable, on a CPU one epoch takes 0.5–2 min); `EPOCHS`; `INPUT_SIZE`; `TRAINABLE` — which layers are trained; `LEARNING_RATE`; `EXPORT_ONNX` — save the model in the ONNX format for deployment on an on-board computer.

**What to look at.** The curves of the training loss and of the test F1 over the epochs; the comparison table with HOG + SVM.

In [ ]:
#@title 4.3 MobileNet transfer learning (optional) { display-mode: "form" }
RUN_MOBILENET = False #@param {type:"boolean"}
EPOCHS = 5 #@param {type:"slider", min:1, max:20, step:1}
INPUT_SIZE = 96 #@param [64, 96, 128, 160] {type:"raw"}
TRAINABLE = "classifier layer only" #@param ["classifier layer only", "classifier + last 2 blocks"]
LEARNING_RATE = 0.002 #@param {type:"number"}
EXPORT_ONNX = False #@param {type:"boolean"}

MOBILENET_RESULT = None
if RUN_MOBILENET:
    import torch
    from torch import nn
    from torchvision import models
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def to_tensor(idx):
        x = np.stack([cv2.resize(PATCHES[i], (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_LINEAR) for i in idx])
        x = torch.from_numpy(x).float().div(255.0).unsqueeze(1).repeat(1, 3, 1, 1)
        return (x - mean) / std

    X_tr, X_te = to_tensor(train_idx), to_tensor(test_idx)
    y_tr, y_te = torch.from_numpy(y_all[train_idx]).long(), y_all[test_idx]
    network = models.mobilenet_v3_small(weights=weights)
    for p in network.parameters():
        p.requires_grad = False
    network.classifier[-1] = nn.Linear(network.classifier[-1].in_features, len(CLASSES))
    trainable = list(network.classifier.parameters())
    if TRAINABLE.startswith("classifier +"):
        for block in network.features[-2:]:
            for p in block.parameters():
                p.requires_grad = True
                trainable.append(p)
    for p in network.classifier.parameters():
        p.requires_grad = True
    network = network.to(device)
    optimizer = torch.optim.Adam(trainable, lr=LEARNING_RATE)
    weight = torch.tensor(len(y_tr) / (len(CLASSES) * np.bincount(y_tr.numpy(), minlength=len(CLASSES)).clip(1)), dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=weight.to(device))      # balanced classes, as class_weight in the SVM

    def predict(X):
        network.eval()
        out = []
        with torch.no_grad():
            for k in range(0, len(X), 256):
                out.append(network(X[k:k + 256].to(device)).argmax(1).cpu())
        return torch.cat(out).numpy()

    history = []
    t0 = time.time()
    for epoch in range(EPOCHS):
        network.train()
        network.features.eval()                                  # statistics of BatchNorm stay those of ImageNet
        perm = torch.randperm(len(X_tr))
        total = 0.0
        for k in range(0, len(perm), 64):
            b = perm[k:k + 64]
            xb, yb = X_tr[b], y_tr[b]
            flip = torch.rand(len(b)) < 0.5                      # augmentation: horizontal flip
            xb[flip] = xb[flip].flip(3)
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(network(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(b)
        pred = predict(X_te)
        history.append(dict(epoch=epoch + 1, loss=total / len(X_tr), f1=f1_score(y_te, pred, average="macro"),
                            accuracy=accuracy_score(y_te, pred)))
        print(f"epoch {epoch + 1}: loss {history[-1]['loss']:.4f}, test F1 {history[-1]['f1']:.3f}, {time.time() - t0:.0f} s")
    history = pd.DataFrame(history)
    MOBILENET_PRED = predict(X_te)
    MOBILENET_RESULT = {"Features": f"MobileNetV3 Small ({TRAINABLE}, {INPUT_SIZE} px, {EPOCHS} epochs)",
                        **scores(y_te, MOBILENET_PRED)}
    comparison = pd.concat([CLASSIFICATION[list(scores(y_te, MOBILENET_PRED))].reset_index(),
                            pd.DataFrame([MOBILENET_RESULT])]).set_index("Features")
    display(comparison.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].plot(history.epoch, history.loss, marker="o", color=COLORS[0])
    axes[0].set(xlabel="epoch", ylabel="training loss", title="MobileNet: training loss")
    axes[1].plot(history.epoch, history.f1, marker="o", color=COLORS[1], label="MobileNet")
    axes[1].axhline(CLASSIFICATION.loc["HOG", "F1 (macro)"], color=COLORS[2], ls="--", label="HOG + SVM")
    axes[1].set(xlabel="epoch", ylabel="test F1 (macro)", title="MobileNet against HOG + SVM"); axes[1].legend()
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    save_figure(fig, "4_3_mobilenet.png")
    network = network.cpu().eval()
    torch.save(network.state_dict(), os.path.join(DIRS["models"], "mobilenet_hituav_patches.pt"))
    if EXPORT_ONNX:
        try:
            torch.onnx.export(network, torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE), os.path.join(DIRS["models"], "mobilenet_hituav_patches.onnx"),
                              input_names=["images"], output_names=["logits"], opset_version=17,
                              dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}})
            print("ONNX model saved in", DIRS["models"])
        except Exception as error:
            print("ONNX export failed (install the packages onnx and onnxscript):", type(error).__name__, error)
else:
    print("The MobileNet block is switched off. Set RUN_MOBILENET in the form to run it.")

**Task 4.**
1. Explain the difference between the accuracy and the macro-averaged F1 using the table of 4.1 (number of patches per class). Which class has the largest share of errors and with which class is it confused?
2. Why do HOG features work better than raw pixels on thermal patches? Compare the columns "F1 day" and "F1 night".
3. Run 4.1 and 4.2 with `TIME_OF_DAY = day only` and then with `night only`. Under which conditions are the thermal signatures easier to separate, and what does it mean for a model that was trained under one condition and used under the other?
4. Read the learning curve: would more labelled images improve the classifier noticeably?

---
## Topic 5. Detection, segmentation and tracking

**Segmentation** assigns a class to every pixel (a mask), **detection** finds objects and outlines them by boxes, **tracking** links the detections of successive frames into trajectories with stable identifiers. In this topic the classical chain *mask → boxes → evaluation against the annotation* is built on thermal frames, then a real vehicle trajectory is filtered with a Kalman filter; the optional blocks use the neural detector YOLO11 and the tracker ByteTrack.

### 5.1. Mask and boxes of warm objects in a thermal frame

**Purpose.** Find warm objects (people, vehicles) in a HIT-UAV thermal frame (white-hot: warm = bright) by two classical methods and evaluate them against the manual annotation.

| Method | Principle |
|---|---|
| Otsu threshold | one global threshold that best separates the histogram into two classes; the mask is cleaned by morphological opening and closing |
| Top-hat + $k\sigma$ | the white top-hat (frame minus its morphological opening) keeps bright details smaller than the structuring element; threshold $\mu + k\sigma$ of the top-hat. The element size follows the expected size of a person in pixels from the ground sampling distance (GSD) |

The GSD at the frame centre is $\mathrm{GSD} = \dfrac{2\,D \tan(\alpha/2)}{N}$, m/px, where $D = h/\sin\beta$ is the distance to the ground at altitude $h$ and camera angle $\beta$, $\alpha$ the horizontal field of view (25°) and $N = 640$ px the frame width. Altitude and angle are coded in the file name of every HIT-UAV image.

A detection is **correct** (TP) when its centre lies inside an annotated box of a target class; a detection inside a box of another class is counted apart ("other"); the remaining detections are false alarms (FP).

**Parameters.** `IMAGE_FILE` — thermal frame; `TARGET` — target classes; `K_STD` — threshold $k$ of the top-hat method; `MORPH_KERNEL`, `MIN_AREA` — cleaning of the Otsu mask; `N_EVAL_IMAGES` — number of day and of night test images for the precision–recall curves.

**What to look at.** The binary masks; TP (green), FP (red), "other" (orange) and missed objects (white); the **precision–recall curves** over $k$ for day and night: at night the background is cooler and warm objects are more contrasted.

In [ ]:
#@title 5.1 Mask and boxes of warm objects (HIT-UAV) { display-mode: "form" }
IMAGE_FILE = "1_60_30_0_00047.jpg · night, 60 m, 30°, 18 persons" #@param ["1_60_30_0_00047.jpg · night, 60 m, 30°, 18 persons", "0_60_60_0_01734.jpg · day, 60 m, 60°, 9 persons", "1_70_90_0_07776.jpg · night, 70 m, 90°, persons, car, bicycles", "0_100_50_0_08162.jpg · day, 100 m, 50°, 15 cars", "1_120_90_0_06161.jpg · night, 120 m, 90°, 9 cars", "0_130_40_0_05603.jpg · day, 130 m, 40°, 6 cars", "1_90_70_0_04990.jpg · night, 90 m, 70°, cars and bicycles", "0_110_80_0_08518.jpg · day, 110 m, 80°, bicycles"]
TARGET = "Person" #@param ["Person", "Person + Bicycle", "all classes"]
K_STD = 5.0 #@param {type:"slider", min:1.0, max:12.0, step:0.5}
MORPH_KERNEL = 3 #@param {type:"slider", min:1, max:9, step:2}
MIN_AREA = 6 #@param {type:"slider", min:1, max:100, step:1}
N_EVAL_IMAGES = 30 #@param {type:"slider", min:10, max:100, step:10}

TARGETS = {"Person": ("Person",), "Person + Bicycle": ("Person", "Bicycle"),
           "all classes": ("Person", "Car", "Bicycle", "OtherVehicle")}[TARGET]


def geometry(file):
    _, h_m, angle, _ = file.split("_")[:4]
    return int(h_m), int(angle)


def detect_otsu(img, kernel=MORPH_KERNEL, min_area=MIN_AREA):
    g = cv2.GaussianBlur(img, (3, 3), 0)
    _, mask = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    se = np.ones((kernel, kernel), np.uint8)
    mask = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_OPEN, se), cv2.MORPH_CLOSE, se)
    n, _, st, _ = cv2.connectedComponentsWithStats(mask, 8)
    boxes = [[x, y, x + w, y + h] for x, y, w, h, a in st[1:] if a >= min_area]
    return np.array(boxes, float).reshape(-1, 4), mask


def detect_tophat(img, file, k=K_STD):
    h_m, angle = geometry(file)
    return aitbc.detect_warm_objects(img, hit.gsd(h_m, angle), k=k)


file = IMAGE_FILE.split(" ")[0]
thermal, t_objects = hit.load(file)
h_m, angle = geometry(file)
gsd = hit.gsd(h_m, angle)
results, panels = [], []
for name, (boxes, mask) in [("Otsu threshold", detect_otsu(thermal)), (f"Top-hat + {K_STD}σ", detect_tophat(thermal, file))]:
    m = aitbc.match_detections(boxes, t_objects, TARGETS)
    results.append({"Method": name, "Detections": len(boxes), "TP": len(m["tp"]), "FP": len(m["fp"]),
                    "Other class": len(m["other"]), "Missed": len(m["missed"]),
                    "Precision": m["precision"], "Recall": m["recall"]})
    out = aitbc.draw_boxes(thermal, [o for o in t_objects if o["cls"] in TARGETS], color=(255, 255, 255), labels=False)
    for group, color in [("tp", (40, 220, 60)), ("fp", (255, 40, 40)), ("other", (255, 150, 0))]:
        out = aitbc.draw_boxes(out, [dict(cls="", box=b) for b in m[group]], color=color, labels=False, thickness=2)
    panels += [mask, out]
display(pd.DataFrame(results).set_index("Method").round(3))
print(f"{file}: {'night' if file[0] == '1' else 'day'}, altitude {h_m} m, camera angle {angle}°, GSD {gsd * 100:.1f} cm/px, "
      f"expected person size {aitbc.expected_size_px(gsd):.1f} px; annotated: "
      + ", ".join(f"{c} {sum(o['cls'] == c for o in t_objects)}" for c in ("Person", "Car", "Bicycle") if any(o['cls'] == c for o in t_objects)))
show_images(panels, ["Otsu: mask", "Otsu: TP green, FP red, other orange, annotated targets white",
                     f"Top-hat + {K_STD}σ: mask", "Top-hat: TP green, FP red, other orange"],
            "5_1_warm_objects.png", cols=2, width=6.0)

# precision-recall curves of the top-hat detector over k, day and night
tp_pool = hit.images[(hit.images.split == "test") & (hit.images[list(t for t in TARGETS if t in hit.images)].sum(axis=1) > 0)]
eval_files = {}
for night, label in [(False, "day"), (True, "night")]:
    cand = tp_pool[tp_pool.night == night].file.values
    eval_files[label] = list(np.random.default_rng(SEED).choice(cand, size=min(N_EVAL_IMAGES, len(cand)), replace=False))
hit.prefetch(eval_files["day"] + eval_files["night"])
ks = [2, 3, 4, 5, 6, 8, 10, 12]
pr = []
for label, files in eval_files.items():
    loaded = [(f, *hit.load(f)) for f in files]
    for k in ks:
        tp = fp = n_obj = 0
        for f, img, objs in loaded:
            boxes, _ = detect_tophat(img, f, k)
            m = aitbc.match_detections(boxes, objs, TARGETS)
            tp += len(m["tp"]); fp += len(m["fp"]); n_obj += sum(o["cls"] in TARGETS for o in objs)
        pr.append(dict(time=label, k=k, precision=tp / max(tp + fp, 1), recall=tp / max(n_obj, 1), objects=n_obj))
pr = pd.DataFrame(pr)
pr.to_csv(os.path.join(OUT, "warm_objects_pr.csv"), index=False)
fig, ax = plt.subplots(figsize=(7, 4.6))
for (label, g), color in zip(pr.groupby("time"), [COLORS[3], COLORS[0]]):
    ax.plot(g.recall, g.precision, marker="o", color=color, label=f"{label} ({len(eval_files[label])} images, {g.objects.iloc[0]} targets)")
    for r in g.itertuples():
        ax.annotate(f"k={r.k}", (r.recall, r.precision), textcoords="offset points", xytext=(4, 4), fontsize=7, color=color)
ax.set(xlabel="recall", ylabel="precision", xlim=(0, 1.02), ylim=(0, 1.02),
       title=f"Top-hat detector: precision–recall over k, target {TARGET}")
ax.grid(alpha=0.3); ax.legend(loc="lower left")
fig.tight_layout()
save_figure(fig, "5_1_pr_curves.png")

### 5.2. Kalman filter for a real vehicle trajectory

**Purpose.** Track the position of a vehicle in the UAV video when the measurements of a detector are noisy and some of them are missing (the object is occluded, the detector failed, a packet of the video link was lost).

The true trajectory is the **manual annotation** of a vehicle of the M3OT sequence chosen in 1.1 (centre of its box, 10 times per second). The "detector measurements" are produced from it by adding Gaussian noise with $\sigma_z$ and deleting a share of the frames at random.

**Model of constant velocity.** State $\mathbf{x} = (x, y, v_x, v_y)^\top$, transition over $\Delta t$:
$$\mathbf{x}_k = F\,\mathbf{x}_{k-1} + \mathbf{w}_k,\quad F = \begin{pmatrix} 1 & 0 & \Delta t & 0\\ 0 & 1 & 0 & \Delta t\\ 0 & 0 & 1 & 0\\ 0 & 0 & 0 & 1 \end{pmatrix},\quad Q = \sigma_a^2\, G G^\top,\ G = \begin{pmatrix} \Delta t^2/2 & 0\\ 0 & \Delta t^2/2\\ \Delta t & 0\\ 0 & \Delta t \end{pmatrix},$$
measurement $\mathbf{z}_k = (x, y)^\top + \mathbf{v}_k$, $R = \sigma_z^2 I$.

**One step:** *prediction* $\hat{\mathbf{x}}^- = F\hat{\mathbf{x}}$, $P^- = F P F^\top + Q$; *correction* (only when a measurement exists) $K = P^- H^\top (H P^- H^\top + R)^{-1}$, $\hat{\mathbf{x}} = \hat{\mathbf{x}}^- + K(\mathbf{z} - H\hat{\mathbf{x}}^-)$, $P = (I - KH)P^-$. In a gap the filter continues the trajectory by the prediction, and its uncertainty $P$ grows.

**Parameters.** `TRACK_RANK` — vehicle (1 = the longest annotated track of the sequence); `MEAS_SIGMA_PX` — noise of the detector $\sigma_z$, px; `DROPOUT` — share of missing measurements; `ACCEL_STD` — $\sigma_a$, px/s²: how much the filter trusts the model of constant velocity (small = smooth but slow reaction to manoeuvres, large = follows the noise); `N_RUNS` — number of random repetitions for the curves.

**What to look at.** The trajectory over the frame; the error against time with the $2\sigma$ band of the filter (gaps are shaded); the curves RMSE(dropout) and RMSE($\sigma_a$) — the second one has a minimum: this is the tuning of the filter.

In [ ]:
#@title 5.2 Kalman filter for a real vehicle trajectory (M3OT) { display-mode: "form" }
TRACK_RANK = 1 #@param {type:"slider", min:1, max:10, step:1}
MEAS_SIGMA_PX = 4.0 #@param {type:"slider", min:0.5, max:15, step:0.5}
DROPOUT = 0.25 #@param {type:"slider", min:0.0, max:0.7, step:0.05}
ACCEL_STD = 20.0 #@param {type:"slider", min:1, max:300, step:1}
N_RUNS = 20 #@param {type:"slider", min:5, max:100, step:5}

lengths = TRACKS.groupby("id").size().sort_values(ascending=False)
track_id = int(lengths.index[min(TRACK_RANK, len(lengths)) - 1])
trk = TRACKS[TRACKS.id == track_id].sort_values("frame")
t_trk = (trk.frame.values - 1) / FPS
truth = trk[["cx", "cy"]].values


def simulate(truth, sigma, dropout, seed):
    g = np.random.default_rng(seed)
    z = truth + g.normal(0, sigma, truth.shape)
    available = g.random(len(truth)) >= dropout
    available[0] = True
    return z, available


def kalman_track(t, z, available, sigma_z, sigma_a):
    x = np.array([z[0, 0], z[0, 1], 0.0, 0.0])
    P = np.diag([sigma_z ** 2, sigma_z ** 2, 50.0 ** 2, 50.0 ** 2])
    Hm = np.array([[1.0, 0, 0, 0], [0, 1.0, 0, 0]])
    R = np.eye(2) * sigma_z ** 2
    est, var = [x[:2].copy()], [P[0, 0] + P[1, 1]]
    for k in range(1, len(t)):
        dt = t[k] - t[k - 1]
        F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1.0]])
        G = np.array([[dt ** 2 / 2, 0], [0, dt ** 2 / 2], [dt, 0], [0, dt]])
        x = F @ x
        P = F @ P @ F.T + sigma_a ** 2 * G @ G.T
        if available[k]:
            S = Hm @ P @ Hm.T + R
            K = P @ Hm.T @ np.linalg.inv(S)
            x = x + K @ (z[k] - Hm @ x)
            P = (np.eye(4) - K @ Hm) @ P
        est.append(x[:2].copy()); var.append(P[0, 0] + P[1, 1])
    return np.array(est), np.array(var)


def hold_last(z, available):
    out = z.copy()
    for k in range(1, len(z)):
        if not available[k]:
            out[k] = out[k - 1]
    return out


rmse = lambda a, b: float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=1))))
z, available = simulate(truth, MEAS_SIGMA_PX, DROPOUT, SEED)
est, var = kalman_track(t_trk, z, available, MEAS_SIGMA_PX, ACCEL_STD)
raw = hold_last(z, available)
err = np.linalg.norm(est - truth, axis=1)
speed = np.linalg.norm(np.diff(truth, axis=0), axis=1) / np.diff(t_trk)
display(pd.DataFrame([
    ("Track", f"id {track_id} of sequence {SEQ}, frames {trk.frame.min()}–{trk.frame.max()} ({len(trk)} positions, {t_trk[-1] - t_trk[0]:.1f} s)"),
    ("Mean image speed of the vehicle", f"{speed.mean():.1f} px/s"),
    ("Missing measurements", f"{(~available).sum()} of {len(available)} ({100 * (~available).mean():.0f} %)"),
    ("RMSE of the measurements (last value held in gaps)", f"{rmse(raw, truth):.2f} px"),
    ("RMSE of the Kalman estimate", f"{rmse(est, truth):.2f} px ({100 * (1 - rmse(est, truth) / rmse(raw, truth)):+.0f} % against the measurements)"),
    ("Share of time inside the 2σ band of the filter", f"{100 * np.mean(err <= 2 * np.sqrt(var)):.0f} %"),
], columns=["Quantity", "Value"]).set_index("Quantity"))

mid_frame = int(trk.frame.values[len(trk) // 2])
background = m3ot.frames(SEQ, CH, start=mid_frame - 1, count=1)[0]
fig = plt.figure(figsize=(15, 8.4))
grid = fig.add_gridspec(2, 2, width_ratios=[1.05, 1])
ax = fig.add_subplot(grid[:, 0])
ax.imshow(background)
ax.plot(truth[:, 0], truth[:, 1], color="w", lw=2.2, label="annotation (truth)")
ax.scatter(z[available, 0], z[available, 1], s=9, color="#ffb000", alpha=0.8, label="noisy measurements")
ax.plot(est[:, 0], est[:, 1], color="#00d0ff", lw=1.6, label="Kalman estimate")
pad = 60
ax.set_xlim(max(truth[:, 0].min() - pad, 0), min(truth[:, 0].max() + pad, background.shape[1]))
ax.set_ylim(min(truth[:, 1].max() + pad, background.shape[0]), max(truth[:, 1].min() - pad, 0))
ax.set_title(f"Vehicle {track_id}, sequence {SEQ}, background frame {mid_frame}"); ax.legend(loc="best"); ax.axis("off")
ax = fig.add_subplot(grid[0, 1])
ax.plot(t_trk, np.linalg.norm(raw - truth, axis=1), color="#ffb000", lw=1, label="measurement error")
ax.plot(t_trk, err, color=COLORS[0], lw=1.6, label="Kalman error")
ax.fill_between(t_trk, 0, 2 * np.sqrt(var), color=COLORS[0], alpha=0.15, label="2σ band of the filter")
for k in np.where(~available)[0]:
    ax.axvspan(t_trk[k] - 0.5 / FPS, t_trk[k] + 0.5 / FPS, color="grey", alpha=0.15, lw=0)
ax.set(xlabel="time, s", ylabel="position error, px", title="Error against time (grey = missing measurements)")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)

drops = np.arange(0, 0.71, 0.1)
accels = np.geomspace(1, 1000, 13)
curve_d = np.array([[(rmse(hold_last(*simulate(truth, MEAS_SIGMA_PX, d, s)), truth),
                      rmse(kalman_track(t_trk, *simulate(truth, MEAS_SIGMA_PX, d, s), MEAS_SIGMA_PX, ACCEL_STD)[0], truth))
                     for s in range(N_RUNS)] for d in drops]).mean(1)
curve_a = np.array([np.mean([rmse(kalman_track(t_trk, *simulate(truth, MEAS_SIGMA_PX, DROPOUT, s), MEAS_SIGMA_PX, a)[0], truth)
                             for s in range(N_RUNS)]) for a in accels])
ax = fig.add_subplot(grid[1, 1])
ax.plot(100 * drops, curve_d[:, 0], marker="o", color="#ffb000", label="measurements, last value held")
ax.plot(100 * drops, curve_d[:, 1], marker="s", color=COLORS[0], label=f"Kalman, σa = {ACCEL_STD:g} px/s²")
ax.set(xlabel="missing measurements, %", ylabel=f"RMSE, px (mean of {N_RUNS} runs)", title="Robustness to missing measurements")
ax.grid(alpha=0.3); ax.legend(loc="upper left")
inset = ax.inset_axes([0.62, 0.12, 0.35, 0.42])
inset.semilogx(accels, curve_a, color=COLORS[1], marker=".")
inset.axvline(ACCEL_STD, color="grey", ls=":")
best_a = accels[np.argmin(curve_a)]
inset.set_title(f"RMSE against σa (min at {best_a:.0f})", fontsize=8); inset.tick_params(labelsize=7)
fig.tight_layout()
save_figure(fig, "5_2_kalman.png")
TRACKING_RESULT = dict(sequence=SEQ, track_id=track_id, positions=len(trk), meas_sigma_px=MEAS_SIGMA_PX, dropout=DROPOUT,
                       accel_std=ACCEL_STD, rmse_measurements_px=rmse(raw, truth), rmse_kalman_px=rmse(est, truth),
                       best_accel_std=float(best_a))

### 5.3. YOLO11: neural detection in a drone frame (optional block)

**Purpose.** Run the pre-trained detector YOLO11 (trained on the COCO data set of ordinary photographs) on the VisDrone frame from 2.1 and evaluate it against the manual annotation.

The COCO classes are mapped to the VisDrone classes: *person* → pedestrian and people; *car* → car and van; *truck*, *bus* → truck and bus; *bicycle*, *motorcycle* → bicycle and motor. A detection is correct when its intersection over union with an unmatched annotated box of the same group is at least `IOU_THRESHOLD`:
$$\mathrm{IoU} = \frac{|A \cap B|}{|A \cup B|}.$$

**Parameters.** `RUN_YOLO`; `YOLO_MODEL` — `n` (2.6 M parameters, fast) or `s` (9.4 M, more accurate); `CONFIDENCE` — minimum confidence of a detection; `IMAGE_SIZE` — input size of the network (letterbox, block 2.3); `IOU_THRESHOLD`; `SWEEP_IMAGE_SIZE` — repeat the evaluation for several input sizes.

**What to look at.** The table per class group; missed objects (yellow): mostly small and dense objects; the curve recall($S$): an input size that fits the small objects improves the recall without any training. The **confidence of the model is not its accuracy**: the accuracy is known only from the comparison with an annotation.

In [ ]:
#@title 5.3 YOLO11 on the drone frame (optional) { display-mode: "form" }
RUN_YOLO = False #@param {type:"boolean"}
YOLO_MODEL = "yolo11n.pt" #@param ["yolo11n.pt", "yolo11s.pt"]
CONFIDENCE = 0.25 #@param {type:"slider", min:0.05, max:0.9, step:0.05}
IMAGE_SIZE = 1280 #@param [640, 960, 1280, 1600] {type:"raw"}
IOU_THRESHOLD = 0.4 #@param {type:"slider", min:0.2, max:0.8, step:0.05}
SWEEP_IMAGE_SIZE = True #@param {type:"boolean"}

YOLO_RESULT = None
GROUPS_GT = {"persons": ("pedestrian", "people"), "cars": ("car", "van"), "trucks and buses": ("truck", "bus"),
             "two-wheelers": ("bicycle", "motor")}
GROUPS_COCO = {"persons": ("person",), "cars": ("car",), "trucks and buses": ("truck", "bus"),
               "two-wheelers": ("bicycle", "motorcycle")}


def evaluate_detections(dets, objects, iou_min=IOU_THRESHOLD):
    rows, tp_boxes, fp_boxes, missed = [], [], [], []
    for group in GROUPS_GT:
        gt = [o["box"] for o in objects if o["cls"] in GROUPS_GT[group]]
        pr = sorted([d for d in dets if d["cls"] in GROUPS_COCO[group]], key=lambda d: -d["conf"])
        used, tp = set(), 0
        for d in pr:
            ious = [aitbc.box_iou(d["box"], g) if j not in used else 0.0 for j, g in enumerate(gt)]
            j = int(np.argmax(ious)) if ious else -1
            if j >= 0 and ious[j] >= iou_min:
                used.add(j); tp += 1; tp_boxes.append(d["box"])
            else:
                fp_boxes.append(d["box"])
        missed += [g for j, g in enumerate(gt) if j not in used]
        rows.append({"Group": group, "Annotated": len(gt), "Detections": len(pr), "TP": tp,
                     "Precision": tp / len(pr) if pr else np.nan, "Recall": tp / len(gt) if gt else np.nan})
    return pd.DataFrame(rows).set_index("Group"), tp_boxes, fp_boxes, missed


if RUN_YOLO:
    if importlib.util.find_spec("ultralytics") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    from ultralytics import YOLO
    detector = YOLO(YOLO_MODEL)
    frame_bgr = cv2.cvtColor(full_rgb, cv2.COLOR_RGB2BGR)

    def run_yolo(size):
        t0 = time.time()
        r = detector.predict(frame_bgr, imgsz=size, conf=CONFIDENCE, verbose=False)[0]
        dt = time.time() - t0
        dets = [dict(cls=r.names[int(c)], conf=float(p), box=[float(v) for v in b])
                for b, c, p in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy())]
        return dets, dt

    detector.predict(frame_bgr, imgsz=IMAGE_SIZE, conf=CONFIDENCE, verbose=False)        # warm-up
    dets, dt = run_yolo(IMAGE_SIZE)
    table, tp_b, fp_b, missed_b = evaluate_detections(dets, full_objects)
    total = table[["Annotated", "Detections", "TP"]].sum()
    table.loc["all groups"] = [total.Annotated, total.Detections, total.TP, total.TP / max(total.Detections, 1), total.TP / max(total.Annotated, 1)]
    display(table.round(3))
    table.to_csv(os.path.join(OUT, "yolo_evaluation.csv"))
    out = aitbc.draw_boxes(full_rgb, [dict(cls="", box=b) for b in missed_b], color=(255, 230, 0), labels=False)
    out = aitbc.draw_boxes(out, [dict(cls="", box=b) for b in fp_b], color=(255, 40, 40), labels=False, thickness=2)
    out = aitbc.draw_boxes(out, [dict(cls="", box=b) for b in tp_b], color=(40, 220, 60), labels=False, thickness=2)
    show_images([out], [f"{YOLO_MODEL}, S = {IMAGE_SIZE}, conf ≥ {CONFIDENCE}: TP green, FP red, missed yellow "
                        f"({dt * 1000:.0f} ms per frame)"], "5_3_yolo.png", width=14)
    YOLO_RESULT = dict(model=YOLO_MODEL, image=IMAGE_NAME, image_size=IMAGE_SIZE, confidence=CONFIDENCE,
                       precision=float(table.loc["all groups", "Precision"]), recall=float(table.loc["all groups", "Recall"]))
    if SWEEP_IMAGE_SIZE:
        sweep_rows = []
        for size in [640, 960, 1280, 1600, 1920]:
            d, dt_s = run_yolo(size)
            tb, *_ = evaluate_detections(d, full_objects)
            s = tb[["Annotated", "Detections", "TP"]].sum()
            sweep_rows.append(dict(size=size, recall=s.TP / max(s.Annotated, 1), precision=s.TP / max(s.Detections, 1), ms=1000 * dt_s))
        sweep_rows = pd.DataFrame(sweep_rows)
        fig, ax = plt.subplots(figsize=(7.5, 3.8))
        ax.plot(sweep_rows["size"], sweep_rows.recall, marker="o", color=COLORS[0], label="recall")
        ax.plot(sweep_rows["size"], sweep_rows.precision, marker="s", color=COLORS[1], label="precision")
        ax.set(xlabel="input size S, px", ylabel="share", ylim=(0, 1), title=f"{YOLO_MODEL} on the drone frame against the input size")
        ax2 = ax.twinx(); ax2.spines["right"].set_visible(True)
        ax2.bar(sweep_rows["size"], sweep_rows.ms, width=60, color="grey", alpha=0.3)
        ax2.set_ylabel("time per frame, ms")
        ax.legend(loc="upper left"); ax.grid(alpha=0.3)
        fig.tight_layout()
        save_figure(fig, "5_3_yolo_input_size.png")
else:
    print("The YOLO block is switched off. Set RUN_YOLO in the form to run it.")

### 5.4. ByteTrack: multi-object tracking in the UAV video (optional block)

**Purpose.** Detect vehicles with YOLO11 in every frame of a real M3OT video and link them into tracks with ByteTrack. ByteTrack predicts every track with a Kalman filter (as in 5.2) and matches the predictions with the detections in two stages: first with confident detections, then with weak ones — this keeps the tracks of partly occluded objects.

**Parameters.** `RUN_BYTETRACK`; `TRACK_SEQUENCE` — visible sequence of M3OT; `TRACK_MODEL`; `TRACK_CONFIDENCE`; `TRACK_IMAGE_SIZE` — vehicles at 100–120 m occupy 10–25 px, therefore the frame (640 × 512) is enlarged to 1920 px at the input of the network; `MAX_TRACK_FRAMES`. On a CPU YOLO11s at 1920 px takes about 1 s per frame, on a T4 GPU about 0.05 s.

The detector was trained on COCO photographs taken from the ground. From 100–120 m a car is a small rectangle seen from above, and the pre-trained network finds only part of the vehicles; on the highway scenes 1-03 and 1-04, where the cars are smallest, it may find none. This is a direct illustration of the **domain gap**: a network for UAV video has to be fine-tuned on aerial images (e.g. VisDrone).

**What to look at.** The trajectories of the tracks over the last frame; the number of detected and of annotated vehicles in every frame; the number of track identifiers against the number of annotated vehicles: extra identifiers mean that tracks were broken and started again (*ID switches*). The annotated video is saved in `outputs`.

In [ ]:
#@title 5.4 ByteTrack on the UAV video (optional) { display-mode: "form" }
RUN_BYTETRACK = False #@param {type:"boolean"}
TRACK_SEQUENCE = "1-05 · day, crossroads" #@param ["1-05 · day, crossroads", "1-07 · day, crossroads", "1-09 · day, city street with parked cars", "1-01 · day, suburban road", "1-04 · day, suburban highway"]
TRACK_MODEL = "yolo11s.pt" #@param ["yolo11n.pt", "yolo11s.pt"]
TRACK_CONFIDENCE = 0.1 #@param {type:"slider", min:0.05, max:0.8, step:0.05}
TRACK_IMAGE_SIZE = 1920 #@param [640, 1280, 1920] {type:"raw"}
MAX_TRACK_FRAMES = 60 #@param {type:"slider", min:10, max:150, step:10}

BYTETRACK_RESULT = None
if RUN_BYTETRACK:
    for module, package in [("ultralytics", "ultralytics"), ("lap", "lap>=0.5.12")]:      # lap: assignment solver of ByteTrack
        if importlib.util.find_spec(module) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    from ultralytics import YOLO
    tracker_model = YOLO(TRACK_MODEL)
    vehicle_classes = [2, 5, 7]                                    # COCO: car, bus, truck
    TSEQ = TRACK_SEQUENCE.split(" ")[0]
    n_proc = int(min(MAX_TRACK_FRAMES, m3ot.n_frames(TSEQ, "rgb")))
    TFRAMES = m3ot.frames(TSEQ, "rgb", start=0, count=n_proc)
    TTRACKS = m3ot.tracks(TSEQ, "rgb")
    out_path = os.path.join(OUT, f"bytetrack_{TSEQ}.mp4")
    rows, annotated = [], []
    t0 = time.time()
    for k in range(n_proc):
        r = tracker_model.track(cv2.cvtColor(TFRAMES[k], cv2.COLOR_RGB2BGR), persist=True, tracker="bytetrack.yaml",
                                conf=TRACK_CONFIDENCE, imgsz=TRACK_IMAGE_SIZE, classes=vehicle_classes, verbose=False)[0]
        annotated.append(cv2.cvtColor(r.plot(line_width=1, font_size=8), cv2.COLOR_BGR2RGB))
        if r.boxes.id is not None:
            for b, tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.cpu().numpy()):
                rows.append(dict(frame=k, id=int(tid), cx=(b[0] + b[2]) / 2, cy=(b[1] + b[3]) / 2))
    elapsed = time.time() - t0
    aitbc.write_video(out_path, annotated, fps=FPS)
    tr = pd.DataFrame(rows, columns=["frame", "id", "cx", "cy"])
    gt_part = TTRACKS[TTRACKS.frame <= n_proc]
    det_per_frame = tr.groupby("frame").size().reindex(range(n_proc), fill_value=0)
    gt_per_frame = gt_part.groupby("frame").size().reindex(range(1, n_proc + 1), fill_value=0)
    display(pd.DataFrame([
        ("Sequence", f"{TSEQ} ({m3ot.SEQUENCES[TSEQ]}), visible channel, {TRACK_MODEL}, S = {TRACK_IMAGE_SIZE}"),
        ("Processed frames", f"{n_proc} ({elapsed / n_proc * 1000:.0f} ms per frame)"),
        ("Track identifiers", tr.id.nunique()),
        ("Annotated vehicles in the same frames", gt_part.id.nunique()),
        ("Mean detected / annotated vehicles per frame", f"{det_per_frame.mean():.1f} / {gt_per_frame.mean():.1f}"),
        ("Annotated video", out_path),
    ], columns=["Quantity", "Value"]).set_index("Quantity"))
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.2, 1]})
    axes[0].imshow(TFRAMES[n_proc - 1])
    cmap = plt.cm.tab20
    for j, (tid, g) in enumerate(tr.groupby("id")):
        axes[0].plot(g.cx, g.cy, color=cmap(j % 20), lw=1.5)
    axes[0].set_title(f"Trajectories of {tr.id.nunique()} tracks over frame {n_proc}"); axes[0].axis("off")
    axes[1].plot(np.arange(n_proc) / FPS, det_per_frame.values, color=COLORS[0], label="tracked by YOLO + ByteTrack")
    axes[1].plot(np.arange(n_proc) / FPS, gt_per_frame.values, color="k", ls="--", label="annotated")
    axes[1].set(xlabel="time, s", ylabel="vehicles in the frame", title="Detected and annotated vehicles")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    fig.tight_layout()
    save_figure(fig, "5_4_bytetrack.png")
    BYTETRACK_RESULT = dict(sequence=TSEQ, model=TRACK_MODEL, image_size=TRACK_IMAGE_SIZE, frames=n_proc, track_ids=int(tr.id.nunique()), annotated_vehicles=int(gt_part.id.nunique()),
                            mean_detected=float(det_per_frame.mean()), mean_annotated=float(gt_per_frame.mean()))
else:
    print("The ByteTrack block is switched off. Set RUN_BYTETRACK in the form to run it.")

**Task 5.**
1. In 5.1 compare the two methods on a day frame and on a night frame. Why does a single global threshold fail when the scene contains warm asphalt or roofs?
2. From the precision–recall curves choose $k$ for night surveillance where a missed person is more costly than a false alarm.
3. In 5.2 set `DROPOUT = 0.6`. How much does the filter gain over the held measurements? Find the minimum of the curve RMSE($\sigma_a$) for `MEAS_SIGMA_PX` = 2 and 10 and explain the shift.
4. (Optional) In 5.3 compare the recall for `IMAGE_SIZE` 640 and 1600 and relate it to the plot of small objects of block 2.3.

---
## Final mini-project

Choose one task according to the available data: preprocessing of degraded frames, classification, segmentation, or detection and tracking. The blocks of this notebook are the templates.

### Requirements to the result
1. Describe the source, the content and the limitations of the data set (sensor, altitude, time of day, annotation).
2. Separate training and test data **before** the choice of the parameters. For video, split by independent recordings or images, not by neighbouring frames of one recording.
3. Save the source data, the parameters, the code and the results in the project structure (`data`, `frames`, `models`, `outputs`).
4. Justify the choice of the metrics. For detection, precision and recall require a reference annotation; the confidence of a model does not replace its accuracy.
5. Explain the typical errors and the limits of applicability of the method.

### Template of the conclusion
**Task:** …
**Data and the way of splitting:** …
**Method and parameters:** …
**Computed metrics:** …
**Typical errors:** …
**Limitations and further changes:** …

The cell below collects the parameters and the main results of this run in `outputs/course_summary.json`.

In [ ]:
#@title Final report of the run { display-mode: "form" }
summary = {
    "seed": SEED, "environment": ENVIRONMENT,
    "video": dict(dataset="M3OT", sequence=SEQ, channel=CHANNEL, first_frame=start + 1, frames=count, fps=FPS, file=VIDEO,
                  sampling_fps=SAMPLING_FPS, saved_frames=int(len(frames_table))),
    "image": IMAGE_SOURCE,
    "restoration": {"noise_sigma": NOISE_SIGMA, "motion_blur_px": MOTION_BLUR_PX, "jpeg_quality": JPEG_QUALITY,
                    "psnr_db": {k: float(v) for k, v in restoration_table["PSNR, dB"].items() if np.isfinite(v)}},
    "classification": {"data": f"HIT-UAV patches, {TIME_OF_DAY}, {len(train_files)} training / {len(test_files)} test images",
                       "patches_train": int(len(train_idx)), "patches_test": int(len(test_idx)),
                       "metrics": {f: {k: float(v) for k, v in CLASSIFICATION.loc[f].items()} for f in CLASSIFICATION.index},
                       "mobilenet": MOBILENET_RESULT},
    "warm_objects": pr.to_dict(orient="records"),
    "tracking": TRACKING_RESULT,
    "yolo": YOLO_RESULT,
    "bytetrack": BYTETRACK_RESULT,
}
path = os.path.join(OUT, "course_summary.json")
with open(path, "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, default=str)
files = sorted(os.listdir(OUT))
display(Markdown(f"**The base route is complete.** {len(files)} files (figures, tables, report) are saved in `{OUT}`."))
print("\n".join(files))

---
## Self-check and sources

### Self-check
- Run the notebook again from the beginning after a restart of the runtime (*Runtime → Restart session and run all*): the numbers must be the same for the same `SEED`.
- Check that the test images were used neither for training nor for the choice of the hyperparameters (4.2: grouped cross-validation on the training images only).
- Change one parameter at a time and keep the others fixed; save the tables of `outputs` before the next change.
- Distinguish the measured result (a table, a metric against the annotation) from an illustration and from an assumption.

### Data sources and licences
The course library reads the files directly from the repositories of the authors; the data are not redistributed with the notebook. When images or results are published, the data sets must be cited.

1. **M3OT.** Nie Z. et al. M3OT: A Multi-Drone Multi-Modality dataset for Multi-Object Tracking. *Scientific Data*, 12, 1927 (2025). https://doi.org/10.1038/s41597-025-06204-0 · Data: https://doi.org/10.6084/m9.figshare.28308887 · Licence CC BY 4.0.
2. **HIT-UAV.** Suo J. et al. HIT-UAV: A high-altitude infrared thermal dataset for Unmanned Aerial Vehicle-based object detection. *Scientific Data*, 10, 227 (2023). https://doi.org/10.1038/s41597-023-02066-6 · Data: https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset · Licence CC BY 4.0.
3. **VisDrone.** Zhu P. et al. Detection and tracking meet drones challenge. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 44(11), 7380–7399 (2022). Data: https://github.com/VisDrone/VisDrone-Dataset · Terms of use: see the data set page.

### Documentation
- [Google Colab: FAQ and resource limits](https://research.google.com/colaboratory/faq.html)
- [OpenCV: image processing tutorials](https://docs.opencv.org/4.x/d2/d96/tutorial_py_table_of_contents_imgproc.html) · [ORB](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html) · [Homography](https://docs.opencv.org/4.x/d9/dab/tutorial_homography.html)
- [scikit-learn: cross-validation with groups](https://scikit-learn.org/stable/modules/cross_validation.html#group-k-fold) · [SVM](https://scikit-learn.org/stable/modules/svm.html)
- [scikit-image: HOG](https://scikit-image.org/docs/stable/auto_examples/features_detection/plot_hog.html)
- [PyTorch: transfer learning](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/) · [Ultralytics: tracking](https://docs.ultralytics.com/modes/track/)

The total volume of 40 hours belongs to the training programme. The notebook is a practical companion of the programme and does not replace the lectures and methodical materials. The optional blocks MobileNet, YOLO and ByteTrack depend on the availability of packages, model weights and computing resources.